# From Prototype to Production: Computer Vision for Autonomous Cars
A hands-on guide: you take one working model and walk it through the six steps a real team goes through before a model is allowed into a car.

**The model you will work on:** a traffic-sign detector for 6 classes (Stop, Speed Limit 30, Speed Limit 50, No Entry, Give Way, Keep Right).
* **v1, the prototype** (*SignSight ADAS*): YOLOv8n trained on one small dataset. Treat it as the model "currently in the car".
* **The candidate:** YOLO11s trained on that dataset **plus** the European *Street Sign Set*, with corrected labels.

The question that runs through the whole notebook: *is the candidate good enough to replace v1 on a real road, and how would you prove it?*
The answer is **not** one accuracy number. By the end you will have a requirement list, a release gate and a scorecard that say SHIP or NO-SHIP, and why.

| Step | Topic | What you do in the notebook |
|---|---|---|
| 1 | Requirements & metrics | Write testable requirements, build the metrics from scratch, price the errors |
| 2 | Data | Audit labels, coverage, imbalance, leakage; version the dataset |
| 3 | Validation | Train, pick an operating point, evaluate by slice, failure mining, **release gate** |
| 4 | Deployment | Latency budget, ONNX / FP16 / INT8, re-validate after optimising |
| 5 | Perception stack | Tracking, calibration & distance, plausibility checks, fallback states |
| 6 | Monitoring & MLOps | Drift detection, shadow mode, labelling queue, model registry, scorecard |
| 6.4 | Simulation | Webots drive with ground-truth distance (`webots_demo/`; CARLA scripts in `carla_demo/` also work) |

**How to use this notebook**
* Run the cells **top to bottom**; later steps reuse results from earlier ones. If a cell fails, fix it and re-run from there.
* Each step starts with **what you will learn** and ends with **check yourself** questions. Try to answer them before reading on.
* Read the outputs, not just the code: every table and plot is there to answer a question.

**Credits.** Prototype adapted from the *SignSight ADAS* notebook by Malak Chartaoui (ENISo). Extra data: *Street Sign Set* by Alessandro Ferrante (CC BY 4.0).
The requirements and costs in this notebook are **illustrative**, chosen for learning, not taken from a real safety case.

## 0. Setup
**Before you start**
1. In Colab: *Runtime → Change runtime type → T4 GPU*. Without a GPU, training is far too slow; use pre-trained weights instead.
2. **Data** (`DATA_SOURCE` in the config cell): `"kaggle"` downloads both datasets with your own Kaggle API token (*kaggle.com → Settings → API → Create New Token*).
   Store it as a Colab secret named `KAGGLE_API_TOKEN` (key icon in the left bar), or paste it when the notebook asks; never write a token into a notebook cell.
   `"drive"` reads them from `MyDrive/traffic_sign_data/a` and `/b` if someone shared them with you.
   The corrected labels (`label_fixes.csv`) come with the repository and are applied automatically.
3. **Weights:** the trained models are in the repository's `models/` folder and are picked up automatically, so nothing is trained.
   Training the candidate yourself takes well over an hour on a T4; to do it, set `WEIGHTS_MAIN = None` in the config cell. A good exercise for later.

The first cell downloads the repository (in Colab), installs the libraries (Ultralytics, ONNX, ONNX Runtime) and prints which GPU you have.

In [ ]:
import subprocess, sys, os, shutil, random, time, json, math, hashlib, csv, warnings
from pathlib import Path
from collections import Counter, defaultdict, deque
from dataclasses import dataclass
from itertools import combinations

warnings.filterwarnings("ignore")

# In Colab: download this workshop's repository (trained weights, label fixes, simulation results) and work inside it.
REPO_URL = "https://github.com/OWNER/cv-production-workshop"     # <- the workshop repository
REPO_DIR = Path("/content/cv-production-workshop")
if Path("/content").exists() and not Path("models").exists():
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=False)
    if REPO_DIR.exists():
        os.chdir(REPO_DIR)
print("working folder:", Path.cwd())


def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)


pip_install("ultralytics", "onnx", "onnxslim")
import torch
HAS_GPU = torch.cuda.is_available()
try:
    pip_install("onnxruntime-gpu" if HAS_GPU else "onnxruntime")
except Exception as e:                      # fall back to the CPU build
    print("onnxruntime-gpu install failed, using CPU build:", e)
    pip_install("onnxruntime")

import numpy as np, cv2, pandas as pd, yaml
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import linear_sum_assignment

os.environ["WANDB_DISABLED"] = "true"
print("GPU:", torch.cuda.get_device_name(0) if HAS_GPU else "none (CPU) -> training will be very slow, load weights instead")

In [ ]:
# ----------------------------- CONFIG (edit here) -----------------------------
SEED = 42
TARGET_CLASSES = ["Stop", "Speed Limit 30", "Speed Limit 50", "No Entry", "Give Way", "Keep Right"]
NC = len(TARGET_CLASSES)
STOP = TARGET_CLASSES.index("Stop")

# Data: 'kaggle' (download with the Kaggle API) | 'drive' (folders on Google Drive) | 'local' (any folders)
DATA_SOURCE = os.environ.get("WS_DATA_SOURCE", "kaggle")
DATA_ROOT_DIR = Path(os.environ.get("WS_DATA_ROOT", "/content/data"))            # each dataset goes to DATA_ROOT_DIR/<tag>
DRIVE_DATA_DIR = Path("/content/drive/MyDrive/traffic_sign_data")                 # same layout on Drive: .../a and .../b
DATASETS = {   # tag -> source. Filenames get the tag as prefix, so every result can be sliced by source.
    "a": dict(name="SignSight (original prototype data)", kaggle="ahemateja19bec1025/trafficsignlocalizationdetectionyoloannotated",
              dir=os.environ.get("WS_DATA_A")),
    "b": dict(name="Street Sign Set (Ferrante 2025, CC BY 4.0)", kaggle="ferrantealessandro/street-sign-set",
              hf="AlessandroFerrante/StreetSignSet", dir=os.environ.get("WS_DATA_B")),
}
USE_DATASETS = ["a", "b"]    # ["a"] = the original prototype data only
# Oversampling in the training list: rare positives, and look-alike signs (source class names) that the model confuses with ours
OVERSAMPLE_POS = {"Keep Right": 3}
OVERSAMPLE_NEG = (["warn_left_curve", "warn_right_curve", "warn_sharp_left_curve", "warn_sharp_right_curve",
                   "forb_speed_over_20", "forb_speed_over_40", "mand_go_right", "mand_go_straight_right"], 2)
# Label corrections (a reviewed, versioned layer on top of the raw data, see 2.6). Searched in this order; skipped with a warning if absent.
LABEL_FIXES_CANDIDATES = [os.environ.get("WS_LABEL_FIXES", ""), "label_fixes.csv", "/content/label_fixes.csv",
                          "/content/drive/MyDrive/traffic_sign_data/label_fixes.csv"]
# Source class names -> our class. 'Turn Right' in dataset a is mostly the blue *keep right* sign (see the label audit in 2.6).
CLASS_ALIASES = {"Stop": ["stop", "prio_stop"], "Speed Limit 30": ["speed limit 30", "forb_speed_over_30"],
                 "Speed Limit 50": ["speed limit 50", "forb_speed_over_50"], "No Entry": ["no entry", "forb_no_entry"],
                 "Give Way": ["give way", "prio_give_way"], "Keep Right": ["turn right", "mand_pass_right"]}

# Model
BASE_MODEL = "yolo11s.pt"    # candidate architecture (yolov8n.pt = smaller/faster, for weak embedded hardware)
IMGSZ = 640
EPOCHS = int(os.environ.get("WS_EPOCHS", 50))
BATCH = 16
# Trained weights: if the repository's models/ folder is here, they are used and training is skipped (training takes over 1.5 h on a T4).
# Set the paths yourself to use other weights, or set both to None to train from scratch.
_repo_w = lambda name: str(Path("models", name).resolve()) if Path("models", name).exists() else None
WEIGHTS_MAIN = os.environ.get("WS_WEIGHTS_MAIN") or _repo_w("signs6_y11s_fixed.pt")
TRAIN_MAIN = True                                    # only used when WEIGHTS_MAIN is empty
# "v1" = the currently deployed model = the original prototype: YOLOv8n trained on dataset 'a' only (the provided weights were trained at 640 px).
WEIGHTS_V1 = os.environ.get("WS_WEIGHTS_V1") or _repo_w("signs6_v8n_original.pt")
TRAIN_V1_BASELINE = True
V1_BASE, V1_EPOCHS = "yolov8n.pt", int(os.environ.get("WS_V1_EPOCHS", 8))
V1_IMGSZ = int(os.environ.get("WS_V1_IMGSZ", 640 if WEIGHTS_V1 else 416))
print("candidate weights:", WEIGHTS_MAIN or "none -> will train", "| v1 weights:", WEIGHTS_V1 or "none -> will train")

# Optional / slow features
TRY_TENSORRT = False         # GPU only; adds a few minutes of setup

# Paths
IN_COLAB = Path("/content").exists()
WORK = Path("/content/work") if IN_COLAB else Path("./work").resolve()
WORK.mkdir(parents=True, exist_ok=True)
OUT = WORK / "signs6"        # prepared dataset (YOLO format, 6 classes)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 0 if HAS_GPU else "cpu"
if not HAS_GPU:
    EPOCHS, V1_EPOCHS = min(EPOCHS, 3), min(V1_EPOCHS, 2)
    print("CPU only: epochs reduced. Set WEIGHTS_MAIN / WEIGHTS_V1 to use pre-trained weights.")

In [ ]:
# ----------------------------- helpers -----------------------------
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp"}
PALETTE = {i: tuple(int(v) for v in c) for i, c in enumerate(np.random.RandomState(SEED).randint(60, 255, (NC, 3)))}
_trapz = getattr(np, "trapezoid", None) or np.trapz


def show_grid(imgs, titles=None, cols=4, size=3.5):
    rows = math.ceil(len(imgs) / cols)
    plt.figure(figsize=(cols * size, rows * size))
    for i, im in enumerate(imgs):
        plt.subplot(rows, cols, i + 1)
        plt.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); plt.axis("off")
        if titles:
            plt.title(titles[i], fontsize=8)
    plt.tight_layout(); plt.show()


def draw_boxes(img, items, color=(0, 255, 0), thickness=2):
    """items: list of dict(box=(x1,y1,x2,y2), label=str)"""
    img = img.copy()
    for it in items:
        x1, y1, x2, y2 = map(int, it["box"])
        cv2.rectangle(img, (x1, y1), (x2, y2), color, thickness)
        if it.get("label"):
            cv2.putText(img, it["label"], (x1, max(12, y1 - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1, cv2.LINE_AA)
    return img


def yolo_to_xyxy(x, y, w, h, W, H):
    """normalised YOLO centre/size -> pixel corners"""
    return (x - w / 2) * W, (y - h / 2) * H, (x + w / 2) * W, (y + h / 2) * H


def quality(img):
    """-> grayscale image, brightness (mean grey level), sharpness (variance of the Laplacian)"""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return gray, float(gray.mean()), float(cv2.Laplacian(gray, cv2.CV_64F).var())


def nan0(v):
    return 0.0 if v != v else v


def sha_short(data: bytes, n=10):
    return hashlib.sha256(data).hexdigest()[:n]


def file_sha(path, n=10):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()[:n]


def finish_plot(name=None):
    plt.tight_layout()
    if name:
        plt.savefig(WORK / f"{name}.png", dpi=130)
    plt.show()

# Step 1 — Requirements and metrics
**What you will learn:** how to turn a vague goal into requirements you can test, and how to build the metrics that test them.

A goal like "detect Stop signs" cannot be tested: how often, how far away, in what light, how fast? A **requirement** can, because it fixes
a target, the operating conditions, a threshold and a time budget. The operating conditions are called the **ODD** (operational design domain):
the situations the system is designed for. Outside the ODD the system is not expected to work, but it must *notice* that it is outside.

**Background: the three standards you will hear about**
* **ISO 26262** (functional safety): what happens when hardware or software *fails*.
* **ISO 21448 / SOTIF** (safety of the intended functionality): the system works exactly as designed, but the design is not good enough, e.g. the detector misses a sign in fog. Most perception problems live here.
* **ISO/PAS 8800**: safety of AI components in road vehicles (data, training, validation of ML models).

Below are the six requirements (R1–R6) this notebook uses. Every later step produces evidence for one of them, and the release gate in Step 3 checks them all.

In [ ]:
@dataclass
class Requirement:
    rid: str
    text: str
    metric: str          # key looked up in the measured results
    op: str              # '>=' or '<='
    target: float
    unit: str = ""

    def check(self, value):
        """True / False, or None when the value was not measured."""
        if value is None or (isinstance(value, float) and math.isnan(value)):
            return None
        return value >= self.target if self.op == ">=" else value <= self.target


ODD = ("Operational design domain (illustrative): daytime & dusk, dry or light rain, signs 10-40 m ahead, "
       "European-style signs, forward camera 640x480, urban/rural roads.")

REQS = [
    Requirement("R1", "Stop sign recall at the operating point",                    "stop_recall",           ">=", 0.95),
    Requirement("R2", "Recall of every other sign class",                           "other_min_recall",      ">=", 0.80),
    Requirement("R3", "False detections per 100 background images (no sign present)", "bg_fa_per_100",       "<=", 4.0),   # was 2.0; relaxed by the team (2026-11-13): a false sign alert is a nuisance, not a safety risk, at club speeds
    Requirement("R4", "p95 end-to-end latency",                                     "p95_latency_ms",        "<=", 66.0, "ms"),   # 15 FPS
    Requirement("R5", "Worst F1 over in-ODD degradations (blur, noise, brightness, scale, JPEG)", "worst_odd_f1", ">=", 0.60),
    Requirement("R6", "Distance estimation error (simulation: Webots or CARLA)",    "distance_err_pct",      "<=", 15.0, "%"),
]
print(ODD, "\n")
pd.DataFrame([vars(r) for r in REQS]).set_index("rid")

## 1.1 Metric families
Metrics come in families, from "does it find the box?" up to "is the car safe?". Rule of thumb: **mAP is a training and benchmark metric; safety is written in miss rate, false-alarm rate and latency.**

In [ ]:
pd.DataFrame([
    ("Detection", "IoU, precision, recall, F1",              "Does it find the sign? How often is it wrong?"),
    ("Ranking",   "AP, mAP@0.5, mAP@0.5:0.95",              "Quality across all confidence thresholds / box tightness"),
    ("Per class", "confusion matrix, per-class AP",         "Which classes are confused (30 vs 50)?"),
    ("System",    "latency (p50/p95/p99), FPS, memory",     "Does it fit the hardware budget, *every* frame?"),
    ("Safety",    "miss rate by distance, false alarms/km", "What the requirement is actually written in"),
], columns=["family", "metrics", "answers"]).set_index("family")

## 1.2 Metrics from scratch
The evaluation is written by hand here (about 60 lines) and later checked against Ultralytics' own numbers, so nothing is a black box. Read the code once: it is short.

* **IoU** = overlap / union of the predicted and true box.
* A prediction is a **TP** if IoU ≥ 0.5 *and* the class is right (greedy matching, highest confidence first). Unmatched predictions = **FP**, unmatched truths = **FN**.

In [ ]:
def iou_xyxy(a, b):
    ix1, iy1, ix2, iy2 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    ua = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / ua if ua > 0 else 0.0


def _best_gt(box, gts, skip, iou_thr):
    """Index of the not-skipped ground truth with the highest IoU >= iou_thr, or -1."""
    best, best_iou = -1, iou_thr - 1e-9
    for j, g in enumerate(gts):
        if not skip(j, g) and (v := iou_xyxy(box, g["box"])) > best_iou:
            best, best_iou = j, v
    return best


def match_image(preds, gts, conf_thr, iou_thr=0.5):
    """preds: [{'cls','conf','box'}], gts: [{'cls','box'}].
    Returns dict(keep=[pred idx], pred_tp={idx: bool}, gt_hit=[bool], wrong={pred idx: gt idx})
    'wrong' = unmatched predictions that overlap (IoU>=thr) an unmatched ground truth of ANOTHER class."""
    keep = sorted([i for i, p in enumerate(preds) if p["conf"] >= conf_thr], key=lambda i: -preds[i]["conf"])
    gt_used, pred_tp, wrong = [False] * len(gts), {}, {}
    for i in keep:
        best = _best_gt(preds[i]["box"], gts, lambda j, g: gt_used[j] or g["cls"] != preds[i]["cls"], iou_thr)
        pred_tp[i] = best >= 0
        if best >= 0:
            gt_used[best] = True
    for i in keep:
        if not pred_tp[i]:
            best = _best_gt(preds[i]["box"], gts, lambda j, g: gt_used[j] or j in wrong.values(), iou_thr)
            if best >= 0:
                wrong[i] = best
    return {"keep": keep, "pred_tp": pred_tp, "gt_hit": gt_used, "wrong": wrong}


def per_image_counts(records, thr, iou_thr=0.5):
    """-> array (N_images, NC, 3) with [tp, fp, fn] per image and class."""
    out = np.zeros((len(records), NC, 3))
    for k, r in enumerate(records):
        m = match_image(r["preds"], r["gts"], thr, iou_thr)
        for i in m["keep"]:
            out[k, r["preds"][i]["cls"], 0 if m["pred_tp"][i] else 1] += 1
        for j, hit in enumerate(m["gt_hit"]):
            if not hit:
                out[k, r["gts"][j]["cls"], 2] += 1
    return out


def metrics_from(pic, mask=None):
    """Aggregate a per_image_counts array (optionally only the images where mask is True)."""
    a = pic if mask is None else pic[np.asarray(mask, bool)]
    tp, fp, fn = a.sum(0).T
    TP, FP, FN = tp.sum(), fp.sum(), fn.sum()
    P = lambda t, f: t / (t + f) if (t + f) > 0 else float("nan")
    denom = 2 * TP + FP + FN
    return {"precision": P(TP, FP), "recall": P(TP, FN),
            "f1": 2 * TP / denom if denom > 0 else float("nan"),     # = harmonic mean of precision and recall
            "tp": int(TP), "fp": int(FP), "fn": int(FN),
            "cls_recall": [P(tp[c], fn[c]) for c in range(NC)], "cls_precision": [P(tp[c], fp[c]) for c in range(NC)],
            "cls_n_gt": [int(tp[c] + fn[c]) for c in range(NC)]}


def average_precision(conf, tp, n_gt):
    """101-point interpolated AP (COCO / Ultralytics style)."""
    if n_gt == 0 or len(conf) == 0:
        return float("nan"), np.array([]), np.array([])
    order = np.argsort(-np.asarray(conf))
    tp = np.asarray(tp, float)[order]
    ctp, cfp = np.cumsum(tp), np.cumsum(1 - tp)
    rec, prec = ctp / n_gt, ctp / np.maximum(ctp + cfp, 1e-9)
    mrec = np.concatenate([[0.0], rec, [1.0]])
    mpre = np.flip(np.maximum.accumulate(np.flip(np.concatenate([[1.0], prec, [0.0]]))))
    x = np.linspace(0, 1, 101)
    return float(_trapz(np.interp(x, mrec, mpre), x)), prec, rec


def dataset_ap50(records, iou_thr=0.5):
    """Per-class AP@0.5 and mAP@0.5 from cached predictions (records must hold low-confidence predictions)."""
    confs, flags, n_gt = defaultdict(list), defaultdict(list), Counter()
    for r in records:
        n_gt.update(g["cls"] for g in r["gts"])
        m = match_image(r["preds"], r["gts"], 0.0, iou_thr)
        for i in m["keep"]:
            c = r["preds"][i]["cls"]
            confs[c].append(r["preds"][i]["conf"]); flags[c].append(m["pred_tp"][i])
    ap = {c: average_precision(confs[c], flags[c], n_gt[c])[0] for c in range(NC) if n_gt[c] > 0}
    return ap, float(np.nanmean(list(ap.values()))) if ap else float("nan")

### Toy example (no model needed): what IoU 0.5 looks like, and how precision/recall trade off
**Try it:** the four panels shift the predicted box further from the truth. Before running, guess at which shift IoU drops below 0.5.
Then change the shifts in `[0, 10, 20, 35]` and re-run. In the table below, follow how precision falls and recall rises as the threshold goes down.

In [ ]:
gt_box = (100, 100, 160, 160)
fig, ax = plt.subplots(1, 4, figsize=(14, 3.2))
for a, shift in zip(ax, [0, 10, 20, 35]):
    pb = (100 + shift, 100 + shift // 2, 160 + shift, 160 + shift // 2)
    a.add_patch(plt.Rectangle(gt_box[:2], 60, 60, fill=False, ec="green", lw=2, label="truth"))
    a.add_patch(plt.Rectangle(pb[:2], 60, 60, fill=False, ec="red", lw=2, ls="--", label="prediction"))
    a.set_xlim(80, 220); a.set_ylim(220, 80); a.set_aspect("equal")
    v = iou_xyxy(gt_box, pb)
    a.set_title(f"IoU = {v:.2f}  ({'TP' if v >= 0.5 else 'miss'})")
ax[0].legend(fontsize=7); finish_plot()

# 8 toy predictions, sorted by confidence: lowering the threshold raises recall but admits false alarms
toy_conf = np.array([.95, .90, .85, .80, .70, .60, .45, .30])
toy_tp = np.array([1, 1, 1, 0, 1, 0, 1, 0])
toy_ngt = 6
ap, prec, rec = average_precision(toy_conf, toy_tp, toy_ngt)
plt.figure(figsize=(4.5, 3.5)); plt.plot(rec, prec, marker="o"); plt.xlabel("recall"); plt.ylabel("precision")
plt.title(f"Toy precision-recall curve, AP = {ap:.2f}"); plt.xlim(0, 1.02); plt.ylim(0, 1.05); finish_plot()
pd.DataFrame({"conf": toy_conf, "TP": toy_tp, "precision": prec.round(2), "recall": rec.round(2)})

## 1.3 Errors are not equal — put a price on them
The *same* detector can be "better" or "worse" depending on what each mistake costs. Missing a Stop sign is far worse than a phantom Keep Right.
The cost table below is illustrative (arbitrary units; in a real project it comes from a hazard analysis). Step 3 uses it to **choose the confidence threshold**, instead of keeping 0.25 out of habit.

**Try it:** after finishing Step 3, come back, double `miss_other`, and re-run 3.3. Which way does the threshold move, and why?

In [ ]:
COST = {
    "miss_stop": 100,      # a missed Stop sign is a safety hazard
    "miss_other": 30,      # missed speed limit / No Entry / Give Way / Keep Right
    "wrong_class": 60,     # right place, wrong sign (e.g. 30 read as 50)
    "false_stop": 20,      # phantom Stop -> hard braking, rear-end risk
    "false_other": 5,      # phantom other sign -> mostly comfort
}


def expected_cost(records, thr, iou_thr=0.5):
    """Total cost of all mistakes made on `records` at confidence threshold `thr`."""
    parts = Counter()
    for r in records:
        m = match_image(r["preds"], r["gts"], thr, iou_thr)
        wrong_gt = set(m["wrong"].values())
        mistakes = ["wrong_class" if j in wrong_gt else "miss_stop" if r["gts"][j]["cls"] == STOP else "miss_other"
                    for j, hit in enumerate(m["gt_hit"]) if not hit]
        mistakes += ["false_stop" if r["preds"][i]["cls"] == STOP else "false_other"
                     for i in m["keep"] if not (m["pred_tp"][i] or i in m["wrong"])]
        for k in mistakes:
            parts[k] += COST[k]
    return sum(parts.values()), dict(parts)


pd.Series(COST, name="cost").to_frame()

**Check yourself (Step 1)**
1. Rewrite "the car should see speed limits" as a requirement with a target, conditions, a threshold and a time budget.
2. A detector has precision 0.99 and recall 0.60 on Stop signs. Is that good for a car? What about the reverse?
3. Why is R1 (Stop recall ≥ 0.95) stricter than R2 (other signs ≥ 0.80)? Find the answer in the cost table.

# Step 2 — Data
**What you will learn:** how to check a dataset before trusting any number computed on it.

In production, teams improve the *data* more often than the model. Ask five questions of any dataset:
is it **correct** (labels), does it **cover** the conditions in the ODD, is it **balanced**, is the test set **independent** of training, and can you **reproduce** it (version)?

**Background.** Real driving data comes from a fleet of cars, public datasets and simulation. Labelling follows written guidelines and is audited,
because annotators disagree and make mistakes. Data from video must be split **by drive**, not by frame, or the test set ends up full of near-copies of training images.
This step does a small version of each of these checks.

## 2.1 Get the datasets and select the 6 classes
Two public sources: **a** = the prototype's original data (36 sign classes), **b** = *Street Sign Set* (7.3k European street images, 63 sign classes).
Each file name gets its source as a prefix (`a_…`, `b_…`), so every result can later be split by source.
Every sign that is not one of our 6 classes is dropped from the labels and becomes a **hard negative** (e.g. Speed Limit 40/60/70 teaches the model what is *not* a 30 or a 50).
Images with none of our 6 classes stay in the data as **background (negative) images** — they are what lets us measure false alarms (requirement R3).

**What to check in the output:** the table of images and boxes per source and split. Which source gives most of the training data? Which class is the rarest?
The training list also repeats some images (*oversampling*, set in the config): the rare Keep Right class 3×, and signs that look like ours (curve warnings, Speed Limit 20/40) 2×.

In [ ]:
def kaggle_auth():
    pip_install("--upgrade", "kaggle", "six")                      # the new KGAT_ tokens need a recent client
    kdir = Path.home() / ".kaggle"
    if ((kdir / "kaggle.json").exists() or (kdir / "access_token").exists()
            or "KAGGLE_USERNAME" in os.environ or "KAGGLE_API_TOKEN" in os.environ):
        return
    try:                                                # 1) Colab Secrets (key icon in the left bar): name = KAGGLE_API_TOKEN
        from google.colab import userdata
        os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
    except Exception:
        from getpass import getpass                     # 2) paste it: the input is hidden and never saved in the notebook
        tok = getpass("Paste your Kaggle token (starts with KGAT_), or leave empty to upload a kaggle.json: ").strip()
        if tok:
            os.environ["KAGGLE_API_TOKEN"] = tok
        else:                                           # 3) classic kaggle.json file
            from google.colab import files
            up = files.upload()
            kdir.mkdir(exist_ok=True)
            (kdir / "kaggle.json").write_bytes(next(iter(up.values())))
            (kdir / "kaggle.json").chmod(0o600)


def get_dataset_root(tag):
    ds = DATASETS[tag]
    if DATA_SOURCE == "local":
        return Path(ds["dir"] or DATA_ROOT_DIR / tag)
    if DATA_SOURCE == "drive":
        from google.colab import drive
        drive.mount("/content/drive")
        return DRIVE_DATA_DIR / tag
    dst = DATA_ROOT_DIR / tag; dst.mkdir(parents=True, exist_ok=True)
    if any(dst.rglob("*.yaml")):
        return dst
    kaggle_auth()
    r = subprocess.run(["kaggle", "datasets", "download", "-d", ds["kaggle"], "-p", str(dst), "--unzip"], capture_output=True, text=True)
    print(r.stdout[-500:], r.stderr[-500:])
    if r.returncode != 0:
        if not ds.get("hf"):
            raise RuntimeError(f"Kaggle download of {ds['kaggle']} failed, see the message above")
        print("Kaggle failed, trying Hugging Face (slow: ~15k small files, rate-limited)")
        pip_install("huggingface_hub")
        from huggingface_hub import snapshot_download
        snapshot_download(ds["hf"], repo_type="dataset", local_dir=str(dst), max_workers=8)
    return dst


def norm(s):
    return "".join(ch for ch in str(s).lower() if ch.isalnum())


ALIAS = {norm(a): TARGET_CLASSES.index(t) for t, al in CLASS_ALIASES.items() for a in al + [t]}


def read_source(root):
    """-> class names {id: name}, {split: folder with images/ and labels/} (works for the usual Kaggle / Roboflow layouts)"""
    ok = lambda p: ".cache" not in p.parts                          # skip download caches
    ymls = sorted([p for ext in ("*.yaml", "*.yml") for p in root.rglob(ext) if ok(p)],
                  key=lambda p: ("custom_data" not in p.name.lower(), str(p)))
    cfg = next(c for c in (yaml.safe_load(p.read_text(encoding="utf8")) for p in ymls) if isinstance(c, dict) and "names" in c)
    nm = cfg["names"]
    nm = dict(enumerate(nm)) if isinstance(nm, list) else {int(k): v for k, v in nm.items()}
    found = {sp: p for sp in ["train", "valid", "val", "test"]
             if (p := next((q for q in sorted(root.rglob(sp)) if ok(q) and q.is_dir() and (q / "images").is_dir() and (q / "labels").is_dir()), None))}
    return nm, {"train": found["train"], "val": found.get("valid") or found.get("val"), "test": found["test"]}


shutil.rmtree(OUT, ignore_errors=True)
SOURCE_STATS, ORIG_NAMES = [], {}
for tag in USE_DATASETS:
    names, splits = read_source(get_dataset_root(tag))
    src_to_new = {sid: ALIAS[norm(n)] for sid, n in names.items() if norm(n) in ALIAS}
    missing = set(range(NC)) - set(src_to_new.values())
    print(f"[{tag}] {DATASETS[tag]['name']}: {len(names)} classes;", {names[k]: TARGET_CLASSES[v] for k, v in src_to_new.items()},
          "| missing:", [TARGET_CLASSES[c] for c in missing] or "none")
    for split, sdir in splits.items():
        for d in ("images", "labels"):
            (OUT / d / split).mkdir(parents=True, exist_ok=True)
        n_img = n_box = 0
        for ip in sorted((sdir / "images").iterdir()):
            if ip.suffix.lower() not in IMG_EXT:
                continue
            kept, orig = [], set()
            lp = sdir / "labels" / (ip.stem + ".txt")
            for parts in map(str.split, lp.read_text().strip().splitlines() if lp.exists() else []):
                c = int(float(parts[0])) if parts else None
                if c in names:
                    orig.add(str(names[c]))
                if len(parts) >= 5 and c in src_to_new:
                    kept.append(f"{src_to_new[c]} " + " ".join(parts[1:5]))
            shutil.copy2(ip, OUT / "images" / split / f"{tag}_{ip.name}")
            (OUT / "labels" / split / f"{tag}_{ip.stem}.txt").write_text("\n".join(kept))
            n_img += 1; n_box += len(kept)
            if split == "train":
                ORIG_NAMES[f"{tag}_{ip.name}"] = orig
        SOURCE_STATS.append({"source": tag, "split": split, "images": n_img, "target boxes": n_box})

# Apply the label corrections (missing labels found by the review in 2.6)
LABEL_FIXES = None
fix_file = next((Path(c) for c in LABEL_FIXES_CANDIDATES if c and Path(c).exists()), None)
if fix_file:
    LABEL_FIXES = pd.read_csv(fix_file)
    LABEL_FIXES = LABEL_FIXES[LABEL_FIXES.file.str.split("_", n=1).str[0].isin(USE_DATASETS)]
    for r in LABEL_FIXES.itertuples():
        lp = OUT / "labels" / r.split / (Path(r.file).stem + ".txt")
        assert lp.exists(), f"label fix for unknown image {r.file}"
        if r.action == "add":
            old = lp.read_text().strip()
            lp.write_text((old + "\n" if old else "") + f"{TARGET_CLASSES.index(r.cls_name)} {r.x:.6f} {r.y:.6f} {r.w:.6f} {r.h:.6f}")
    print(f"label fixes applied from {fix_file}: {len(LABEL_FIXES)} boxes added", LABEL_FIXES.groupby("split").size().to_dict())
else:
    print("WARNING: label_fixes.csv not found -> using the raw labels (about 136 real signs are unlabelled, see 2.6)")

SOURCE_NAMES = {t: DATASETS[t]["name"].split(" (")[0] for t in USE_DATASETS}


def source_of(path):
    return Path(path).name.split("_", 1)[0]


# Training list with oversampling (repeating a file in the list = showing it more often per epoch)
train_list, rep_stats = [], Counter()
for ip in sorted((OUT / "images" / "train").iterdir()):
    lab = (OUT / "labels" / "train" / (ip.stem + ".txt")).read_text().split()
    cls_here = {TARGET_CLASSES[int(v)] for v in lab[0::5]}
    n = max([OVERSAMPLE_POS.get(c, 1) for c in cls_here] + [1])
    if not cls_here and ORIG_NAMES.get(ip.name, set()) & set(OVERSAMPLE_NEG[0]):
        n = OVERSAMPLE_NEG[1]; rep_stats["look-alike negatives"] += 1
    elif n > 1:
        rep_stats["rare positives"] += 1
    train_list += [str(ip)] * n
TRAIN_TXT = OUT / "train_list.txt"
TRAIN_TXT.write_text("\n".join(train_list))
print(f"training list: {len(train_list)} entries for {len(set(train_list))} images; repeated:", dict(rep_stats))


def write_data_yaml(name, train_txt):
    p = OUT / name
    p.write_text(yaml.safe_dump({"path": str(OUT), "train": str(train_txt), "val": "images/val", "test": "images/test",
                                 "names": dict(enumerate(TARGET_CLASSES))}, sort_keys=False))
    return p


DATA_YAML = write_data_yaml("data.yaml", TRAIN_TXT)
# training list for the v1 prototype: dataset 'a' only
TRAIN_A_TXT = OUT / "train_a.txt"
TRAIN_A_TXT.write_text("\n".join(str(p) for p in sorted((OUT / "images" / "train").glob("a_*"))))
DATA_YAML_A = write_data_yaml("data_a.yaml", TRAIN_A_TXT)
print("prepared:", OUT)
pd.DataFrame(SOURCE_STATS).pivot_table(index="source", columns="split", values=["images", "target boxes"]).astype(int)

## 2.2 Build one table per image and per box
For every image we measure what a *condition* looks like: **brightness**, **sharpness** (variance of the Laplacian), a **perceptual hash** (for leakage checks).
These cheap statistics stand in for the "weather / time of day / blur" metadata that a real fleet dataset would carry.

In [ ]:
def dhash(gray, size=8):
    s = cv2.resize(gray, (size + 1, size), interpolation=cv2.INTER_AREA)
    return int("".join("1" if b else "0" for b in (s[:, 1:] > s[:, :-1]).flatten()), 2)


def hamming(a, b):
    return bin(a ^ b).count("1")


ITEMS, box_rows, img_rows = {}, [], []
for split in ["train", "val", "test"]:
    ITEMS[split] = []
    for ip in sorted((OUT / "images" / split).glob("*")):
        img = cv2.imread(str(ip)) if ip.suffix.lower() in IMG_EXT else None
        if img is None:
            continue
        H, W = img.shape[:2]
        gray, brightness, blur = quality(img)
        q = {"brightness": brightness, "blur": blur, "hash": dhash(gray)}
        gts, lp = [], OUT / "labels" / split / (ip.stem + ".txt")
        for line in (lp.read_text().strip() if lp.exists() else "").splitlines():
            c, x, y, w, h = line.split()[:5]
            c, x, y, w, h = int(c), float(x), float(y), float(w), float(h)
            gts.append({"cls": c, "box": yolo_to_xyxy(x, y, w, h, W, H)})
            box_rows.append({"split": split, "path": str(ip), "cls": c, "cx": x, "cy": y, "w": w, "h": h, "w_px": w * W, "h_px": h * H})
        ITEMS[split].append({"path": str(ip), "W": W, "H": H, "gts": gts, **q, "source": source_of(ip)})
        img_rows.append({"split": split, "path": str(ip), "source": source_of(ip), "n_boxes": len(gts), **q})
IMGS, BOXES = pd.DataFrame(img_rows), pd.DataFrame(box_rows)
BOXES["cls_name"] = BOXES["cls"].map(dict(enumerate(TARGET_CLASSES)))

summary = BOXES.pivot_table(index="cls_name", columns="split", values="cx", aggfunc="count").reindex(TARGET_CLASSES)[["train", "val", "test"]]
summary.loc["(background images)"] = IMGS[IMGS.n_boxes == 0].groupby("split").size()
summary.loc["(all images)"] = IMGS.groupby("split").size()
summary

### Sample entries
Before computing anything else, look at the data itself: a few rows of each table, one raw label file, and one training image per class and source with its ground-truth boxes.

In [ ]:
display(IMGS.sample(5, random_state=SEED))
display(BOXES.sample(5, random_state=SEED)[["split", "path", "cls_name", "cx", "cy", "w", "h", "w_px", "h_px"]])

# one random example per class and source, where that sign is clearly visible (30 px wide up to 60% of the image width)
rs = random.Random(SEED)
samples = []
for c in range(NC):
    for t in USE_DATASETS:
        cand = [it for it in ITEMS["train"] if it["source"] == t
                and any(g["cls"] == c and 30 <= g["box"][2] - g["box"][0] <= 0.6 * it["W"] for g in it["gts"])]
        if cand:
            samples.append((c, rs.choice(cand)))
lp = OUT / "labels" / "train" / (Path(samples[0][1]["path"]).stem + ".txt")
print(f"raw label file {lp.name}  (class cx cy w h, normalised to [0, 1]):\n{lp.read_text()}")
show_grid([draw_boxes(cv2.imread(it["path"]), [{"box": g["box"], "label": TARGET_CLASSES[g["cls"]]} for g in it["gts"]]) for _, it in samples],
          [f"{TARGET_CLASSES[c]} | {SOURCE_NAMES[it['source']]}" for c, it in samples], cols=4)

## 2.3 Define the condition bands (from the **training** set only)
The brightness and blur scales are cut at fixed thresholds taken from train, then the *same* thresholds are reused for validation, test and — in Step 6 — field data.
If each split had its own thresholds, "dark" would mean something different in each one and the comparisons would be meaningless.

In [ ]:
BR_LO, BR_HI = np.percentile(IMGS[IMGS.split == "train"].brightness, [20, 80])
BL_LO = np.percentile(IMGS[IMGS.split == "train"].blur, 20)
SIZE_EDGES = (16, 32)       # sign width in pixels: small < 16 <= medium < 32 <= large


def band_brightness(v):
    return "dark" if v < BR_LO else ("bright" if v > BR_HI else "normal")


def band_blur(v):
    return "blurry" if v < BL_LO else "sharp"


def band_size(w):
    return "small" if w < SIZE_EDGES[0] else ("medium" if w < SIZE_EDGES[1] else "large")


IMGS["bright_band"] = IMGS.brightness.map(band_brightness)
IMGS["blur_band"] = IMGS.blur.map(band_blur)
BOXES["size_band"] = BOXES.w_px.map(band_size)
BOXES = BOXES.merge(IMGS[["path", "brightness", "blur", "bright_band", "blur_band"]], on="path", how="left")
print(f"dark < {BR_LO:.0f} < normal < {BR_HI:.0f} < bright | blurry: Laplacian var < {BL_LO:.0f} | small < {SIZE_EDGES[0]}px <= medium < {SIZE_EDGES[1]}px <= large")

## 2.4 Coverage grid — where are the holes?
A big dataset can still be empty in the cells that matter. Count boxes per (class × size) and per (brightness × size) in **train** and **test**.

**What to check:** find the cells with very few boxes (for example small signs in the dark). Any result the model gets on those cells later is based on almost no evidence.

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(11, 7))
for r, split in enumerate(["train", "test"]):
    b = BOXES[BOXES.split == split]
    for c, (key, title, order) in enumerate([("cls_name", "class", TARGET_CLASSES), ("bright_band", "brightness", ["dark", "normal", "bright"])]):
        t = pd.crosstab(b[key], b["size_band"]).reindex(index=order, columns=["small", "medium", "large"], fill_value=0)
        a = ax[r, c]; a.imshow(t.values, cmap="YlGnBu")
        a.set_xticks(range(3)); a.set_xticklabels(t.columns); a.set_yticks(range(len(t))); a.set_yticklabels(t.index, fontsize=8)
        for (i, j), v in np.ndenumerate(t.values):
            a.text(j, i, int(v), ha="center", va="center", color="red" if v < 10 else "black", fontsize=9)
        a.set_title(f"{split}: boxes by {title} x sign size  (red = fewer than 10)", fontsize=9)
finish_plot("coverage_grid")
print(f"Median sign width {BOXES.w_px.median():.0f}px; {(BOXES.w_px < SIZE_EDGES[0]).mean() * 100:.0f}% of signs are 'small' (<{SIZE_EDGES[0]}px).")

## 2.5 Class imbalance
Imbalance decides *how you report* (per class, never one average) and what you collect next.

In [ ]:
tr = BOXES[BOXES.split == "train"].cls_name.value_counts().reindex(TARGET_CLASSES)
plt.figure(figsize=(7, 3)); plt.bar(tr.index, tr.values, color="tab:blue")
plt.xticks(rotation=25, ha="right"); plt.ylabel("train boxes"); plt.title(f"Imbalance: {tr.max() / tr.min():.1f}x between largest and smallest class"); finish_plot()

## 2.6 Label audit
Bad labels cap the accuracy you can ever reach, and they silently corrupt your metrics. Automatic checks catch the cheap errors; humans review the rest.

In [ ]:
def draw_yolo_box(path, x, y, w, h, label):
    im = cv2.imread(str(path)); H, W = im.shape[:2]
    return draw_boxes(im, [{"box": yolo_to_xyxy(x, y, w, h, W, H), "label": label}], (0, 0, 255))


b = BOXES
tiny = (b.w_px < 4) | (b.h_px < 4)
extreme = (b.w_px / b.h_px.clip(lower=1e-6) > 2.5) | (b.h_px / b.w_px.clip(lower=1e-6) > 2.5)
audit = pd.DataFrame([
    ("box outside image (centre/size not in [0,1])", int(((b.cx < 0) | (b.cx > 1) | (b.cy < 0) | (b.cy > 1) | (b.w <= 0) | (b.h <= 0) | (b.w > 1) | (b.h > 1)).sum())),
    ("tiny box (< 4 px wide or high)", int(tiny.sum())),
    ("extreme aspect ratio (> 2.5 : 1)", int(extreme.sum())),
    ("duplicate boxes (same class, IoU > 0.9)", sum(g1["cls"] == g2["cls"] and iou_xyxy(g1["box"], g2["box"]) > 0.9
                                                    for its in ITEMS.values() for it in its for g1, g2 in combinations(it["gts"], 2))),
    ("images without label file", int(sum(not (OUT / "labels" / Path(p).parent.name / (Path(p).stem + ".txt")).exists() for p in IMGS.path))),
], columns=["check", "count"]).set_index("check")
display(audit)
odd = b[tiny | extreme].head(8)
if len(odd):
    show_grid([draw_yolo_box(r.path, r.cx, r.cy, r.w, r.h, r.cls_name) for r in odd.itertuples()], [Path(p).name[:22] for p in odd.path], cols=4)

**Look at the crops, per class and per source.** Automatic checks do not catch *semantic* label errors.
In source **a** the class originally named *Turn Right* is mostly the blue **keep right** sign, mixed with real turn-right arrows and even a curve-warning triangle:
one class name, several different signs. That is why we call the class *Keep Right* and map source b's `mand_pass_right` to it.

In [ ]:
def class_crops(cls, source, n=10, size=72):
    out = []
    for it in ITEMS["train"]:
        if it["source"] != source:
            continue
        for g in it["gts"]:
            if g["cls"] == cls:
                x1, y1, x2, y2 = map(int, g["box"])
                cr = cv2.imread(it["path"])[max(0, y1):y2, max(0, x1):x2]
                if cr.size:
                    out.append(cv2.resize(cr, (size, size)))
        if len(out) >= n:
            break
    out = out[:n]
    return np.hstack(out + [np.full((size, size, 3), 255, np.uint8)] * (n - len(out)))


rows = [(f"{c} | {SOURCE_NAMES[t]}", class_crops(TARGET_CLASSES.index(c), t)) for c in ["Keep Right", "Give Way"] for t in USE_DATASETS]
plt.figure(figsize=(12, 1.4 * len(rows)))
for i, (title, strip) in enumerate(rows):
    plt.subplot(len(rows), 1, i + 1); plt.imshow(cv2.cvtColor(strip, cv2.COLOR_BGR2RGB)); plt.axis("off"); plt.title(title, fontsize=9, loc="left")
finish_plot("label_crops")

**Model-assisted label review.** A trained model was run over *all* splits, and every confident detection (conf ≥ 0.5) that overlaps no label was listed
(the script is `experiments/mine_missing_labels.py`). A person looked at each of the 230 candidates: 136 were real signs nobody had labelled (67 of them Give Way!),
the other 94 were genuine model errors (curve warnings read as *Keep Right*, Speed Limit 20/70/90, road markings).
The accepted ones live in `label_fixes.csv`, a reviewed and versioned correction layer applied in 2.1 — the raw data is never edited.

Missing labels hurt twice: they teach the model to *ignore* real signs, and they count correct detections as *false alarms* (R3).
Keep this in mind when comparing models: part of any improvement measured on corrected labels comes from the labels, not the model.

In [ ]:
if LABEL_FIXES is not None:
    display(LABEL_FIXES.pivot_table(index="cls_name", columns="split", values="file", aggfunc="count", fill_value=0))
    sample = LABEL_FIXES.sample(min(8, len(LABEL_FIXES)), random_state=SEED)
    show_grid([draw_yolo_box(OUT / "images" / r.split / r.file, r.x, r.y, r.w, r.h, "added: " + r.cls_name) for r in sample.itertuples()],
              [f"{r.split} | {r.file[:20]}" for r in sample.itertuples()], cols=4)

## 2.7 Leakage check — is the test set really *unseen*?
Consecutive video frames (or augmented copies) look almost identical. If a near-duplicate of a test image is in train, the test score is **optimistic**.
We use a perceptual hash (dHash); a Hamming distance ≤ 4 out of 64 bits means "visually the same image".
*Fix in real projects:* split by **drive / location**, never by random frame.

In [ ]:
tr_h = [it["hash"] for it in ITEMS["train"]]
leak = pd.DataFrame([{"split": s, "path": it["path"], "min_hamming": min(hamming(it["hash"], h) for h in tr_h)}
                     for s in ["val", "test"] for it in ITEMS[s]])
plt.figure(figsize=(6, 3))
for s, c in [("val", "tab:orange"), ("test", "tab:green")]:
    plt.hist(leak[leak.split == s].min_hamming, bins=range(0, 33), alpha=.6, label=s, color=c)
plt.axvline(4.5, color="r", ls="--"); plt.xlabel("distance to nearest TRAIN image (bits)"); plt.legend(); plt.title("Near-duplicate check"); finish_plot()
leak["dup"], leak["source"] = leak.min_hamming <= 4, leak.path.map(source_of)
for s in ["val", "test"]:
    n, tot = int(leak[leak.split == s].dup.sum()), int((leak.split == s).sum())
    print(f"{s}: {n}/{tot} images ({n / tot * 100:.0f}%) have a near-duplicate in train")
leak_test = leak[leak.split == "test"]
print("\nshare of test images with a near-duplicate in train, per source:")
print((leak_test.groupby("source").dup.mean() * 100).round(0).rename(SOURCE_NAMES).to_string())
LEAK_TEST_PCT = float(leak_test.dup.mean() * 100)
if LEAK_TEST_PCT > 5:
    print("\n>>> Warning: a noticeable share of the test set is almost identical to training images; test scores below are likely optimistic.")

## 2.8 Rare-event mining
With a fleet you cannot label everything. You *rank* images by how informative they are and label the top of the list first.
Here: rare class + small sign + dark scene = high priority. Look at the top images: would you agree they are the most useful ones to label?

In [ ]:
freq = tr / tr.sum()


def label_priority(it):
    return (sum(1.0 / freq[TARGET_CLASSES[g["cls"]]] for g in it["gts"])
            + 5 * sum((g["box"][2] - g["box"][0]) < SIZE_EDGES[0] for g in it["gts"])
            + 10 * (it["brightness"] < BR_LO))


top = sorted(((label_priority(it), it) for it in ITEMS["train"] if it["gts"]), key=lambda t: -t[0])[:8]
show_grid([draw_boxes(cv2.imread(it["path"]), [{"box": g["box"], "label": TARGET_CLASSES[g["cls"]]} for g in it["gts"]], (0, 200, 255)) for _, it in top],
          [f"priority {s:.0f}" for s, _ in top], cols=4)

## 2.9 Version the dataset
Every result must be traceable to the exact data that produced it. We hash all images + labels into one **dataset ID** and write a manifest.

In [ ]:
def dataset_version():
    h, n_files = hashlib.sha256(), 0
    for split in ["train", "val", "test"]:
        for ip in sorted((OUT / "images" / split).glob("*")):
            lp = OUT / "labels" / split / (ip.stem + ".txt")
            h.update(ip.name.encode()); h.update(ip.read_bytes()); h.update(lp.read_bytes() if lp.exists() else b"")
            n_files += 1
    return h.hexdigest()[:10], n_files


DATASET_ID, n_files = dataset_version()
manifest = {"dataset_id": DATASET_ID, "n_images": n_files, "classes": TARGET_CLASSES, "sources": {t: DATASETS[t] for t in USE_DATASETS},
            "split_counts": summary.loc[TARGET_CLASSES].to_dict(), "near_duplicate_test_pct": round(LEAK_TEST_PCT, 1),
            "condition_bands": {"brightness_lo": float(BR_LO), "brightness_hi": float(BR_HI), "blur_lo": float(BL_LO)}}
(OUT / "manifest.json").write_text(json.dumps(manifest, indent=2, default=str))
print("dataset id:", DATASET_ID, "| images:", n_files)

**Check yourself (Step 2)**
1. Which class is the rarest in the test set? How many test boxes does it have, and how far would recall move if the model missed two more of them?
2. Why do we keep images that contain *other* signs (Speed Limit 40, curve warnings) instead of deleting them?
3. The leakage check found some near-duplicates between train and test. What does that do to the test score, and how would you split the data to avoid it?
4. You add 50 new images and fix 3 labels. Does the dataset ID change? Why does that matter when you compare two models?

# Step 3 — Validation by scenario
**What you will learn:** how to decide, with evidence, whether a model may replace the one in the car.

One number is not evidence. In this step you (1) train or load the models, (2) pick the confidence threshold with **validation** data and the cost table,
(3) evaluate the **test** set by slice (class, sign size, brightness, blur, degradations), (4) look at the failures one by one, and (5) put everything through a **release gate**.
The test set is used once, at the end, and never to tune anything. If you tune on it, its score stops meaning anything.

**Background.** Testing goes from cheap to expensive: offline dataset → replay of recorded drives → simulation → hardware-in-the-loop → real road.
This notebook covers the offline level (and simulation in section 6.4). At every level, a new model must not be worse than the current one on any critical slice, even if its average is better: that is a **regression gate**.

## 3.1 Detector wrapper and training
All later steps (PyTorch, FP16, ONNX, INT8) use the same interface `detector(image) -> [{'box','cls','conf'}]`, so every variant is evaluated by the **same** code.

Two models are prepared here: the **candidate** (YOLO11s on both datasets) and **v1** (the prototype, YOLOv8n on dataset a only), which plays the "model currently in the car".
If you set `WEIGHTS_MAIN` / `WEIGHTS_V1` in the config, the weights are loaded; otherwise they are trained, which takes a long time.

In [ ]:
from ultralytics import YOLO, settings
try:
    settings.update({"wandb": False})
except Exception:
    pass
WORKERS = 0 if os.name == "nt" else 2


class YoloDetector:
    def __init__(self, weights, imgsz=IMGSZ, half=False, device=DEVICE, label=None):
        self.model, self.weights, self.imgsz = YOLO(str(weights)), str(weights), imgsz
        self.half, self.device, self.label = bool(half and HAS_GPU), device, label or Path(str(weights)).stem
        self(np.zeros((416, 416, 3), np.uint8))               # warm-up

    def __call__(self, img, conf=0.01):
        b = self.model.predict(img, imgsz=self.imgsz, conf=conf, half=self.half, device=self.device, verbose=False)[0].boxes
        return [{"box": tuple(float(v) for v in xy), "cls": int(c), "conf": float(s)}
                for xy, c, s in zip(b.xyxy.cpu().numpy(), b.cls.cpu().numpy(), b.conf.cpu().numpy())]


def train_model(base, epochs, imgsz, run_name, data=None):
    m = YOLO(base)
    m.train(data=str(data or DATA_YAML), epochs=epochs, imgsz=imgsz, batch=BATCH, patience=15, seed=SEED,
            project=str(WORK / "runs"), name=run_name, exist_ok=True, device=DEVICE, workers=WORKERS, plots=True,
            fliplr=0.0,                      # no horizontal flip: it would turn 'Keep Right' into 'Keep Left'
            degrees=5, scale=0.5, close_mosaic=min(10, max(1, epochs // 2)))
    return Path(m.trainer.best)

In [ ]:
t0 = time.time()
if TRAIN_MAIN and not WEIGHTS_MAIN:
    WEIGHTS_MAIN = train_model(BASE_MODEL, EPOCHS, IMGSZ, "main")
assert WEIGHTS_MAIN, "Set WEIGHTS_MAIN or TRAIN_MAIN = True"
print("candidate weights:", WEIGHTS_MAIN, f"({time.time() - t0:.0f}s)")

In [ ]:
# The "currently deployed" v1 = the original prototype: YOLOv8n, dataset 'a' only. The candidate must not regress against it.
if TRAIN_V1_BASELINE and not WEIGHTS_V1:
    WEIGHTS_V1 = train_model(V1_BASE, V1_EPOCHS, V1_IMGSZ, "v1", data=DATA_YAML_A if "a" in USE_DATASETS else DATA_YAML)
DET_MAIN = YoloDetector(WEIGHTS_MAIN, IMGSZ, label="candidate")
DET_V1 = YoloDetector(WEIGHTS_V1, V1_IMGSZ, label="v1") if WEIGHTS_V1 else None
print("v1:", WEIGHTS_V1)

## 3.2 Cache predictions once, evaluate many times
We run the detector at a very low confidence (0.01) and store everything. Any threshold, slice or metric afterwards is just arithmetic on that cache.

In [ ]:
def run_detector(detector, items, transform=None, conf=0.01):
    recs = []
    for it in items:
        img = cv2.imread(it["path"])
        if transform is not None:
            img = transform(img)
        recs.append({"path": it["path"], "gts": it["gts"], "preds": detector(img, conf), "brightness": it["brightness"], "blur": it["blur"]})
    return recs


t0 = time.time()
VAL_MAIN, TEST_MAIN = run_detector(DET_MAIN, ITEMS["val"]), run_detector(DET_MAIN, ITEMS["test"])
print(f"cached {len(VAL_MAIN)} val + {len(TEST_MAIN)} test predictions in {time.time() - t0:.0f}s")

# Sanity check: our from-scratch mAP vs Ultralytics' official number (small differences are normal: NMS/IoU settings differ)
ap_own, map_own = dataset_ap50(TEST_MAIN)
official = YOLO(str(WEIGHTS_MAIN)).val(data=str(DATA_YAML), split="test", imgsz=IMGSZ, batch=BATCH, plots=False, verbose=False, device=DEVICE, workers=WORKERS)
print(f"mAP@0.5  ours = {map_own:.3f}   Ultralytics = {official.box.map50:.3f}")

## 3.3 Choose the operating point with a cost model (on **validation**, not test)
The confidence threshold is a safety decision. Lower → fewer missed signs, more phantom ones. The notebook picks the threshold that minimises the *priced* mistakes from Step 1.

**What to check:** the cost curve and where its minimum is. Is the chosen threshold far from Ultralytics' default of 0.25? Is the curve flat or sharp around it (how much does a small change cost)?

In [ ]:
THRS = np.round(np.arange(0.05, 0.96, 0.05), 2)


def choose_threshold(val_records):
    rows = []
    for t in THRS:
        cost, parts = expected_cost(val_records, t)
        m = metrics_from(per_image_counts(val_records, t))
        rows.append({"thr": t, "cost": cost, "precision": m["precision"], "recall": m["recall"], "stop_recall": m["cls_recall"][STOP], **parts})
    df = pd.DataFrame(rows).fillna(0)
    return float(df.loc[df.cost.idxmin(), "thr"]), df


OP_THR, sweep = choose_threshold(VAL_MAIN)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.3))
ax[0].plot(sweep.thr, sweep.precision, label="precision"); ax[0].plot(sweep.thr, sweep.recall, label="recall")
ax[0].plot(sweep.thr, sweep.stop_recall, label="Stop recall", ls="--"); ax[0].legend(); ax[0].set_title("Trade-off (validation set)")
cols = [c for c in COST if c in sweep]
ax[1].stackplot(sweep.thr, [sweep[c] for c in cols], labels=cols); ax[1].set_ylabel("cost"); ax[1].legend(fontsize=7)
ax[1].set_title(f"Priced mistakes -> chosen threshold = {OP_THR}")
for a in ax:
    a.axvline(OP_THR, color="r", ls=":"); a.set_xlabel("confidence threshold")
finish_plot("operating_point")

## 3.4 Evaluate the test set — by slice
`build_pack` computes everything the release gate needs for one model: headline metrics, per-class results, slices (class / size / brightness / blur / source) with **confidence intervals**, and robustness under degradations.

**What to check:** the F1 interval printed under the headline numbers. The test set is small, so the interval is wide: two models whose intervals overlap a lot may not really differ.

In [ ]:
def wilson(k, n, z=1.96):
    if n == 0:
        return (float("nan"), float("nan"))
    p, d = k / n, 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (max(0.0, c - h), min(1.0, c + h))


def degrade(img, kind, level):
    if kind == "brightness":
        return np.clip(img.astype(np.float32) * level, 0, 255).astype(np.uint8)
    if kind == "blur":
        k = int(level) * 2 + 1
        return cv2.GaussianBlur(img, (k, k), 0)
    if kind == "noise":
        return np.clip(img.astype(np.float32) + np.random.RandomState(SEED).normal(0, level, img.shape), 0, 255).astype(np.uint8)
    if kind == "scale":                                          # smaller = the sign looks further away
        small = cv2.resize(img, None, fx=level, fy=level, interpolation=cv2.INTER_AREA)
        return cv2.resize(small, (img.shape[1], img.shape[0]))
    if kind == "jpeg":                                           # camera / ISP / transmission artefacts
        return cv2.imdecode(cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, int(level)])[1], cv2.IMREAD_COLOR)
    return img


# (kind, level, inside the ODD?)  -> only 'inside' conditions count for requirement R5
DEGRADATIONS = [("brightness", 0.3, False), ("brightness", 0.6, True), ("brightness", 1.4, True),
                ("blur", 1, True), ("blur", 2, True), ("blur", 3, False),
                ("noise", 10, True), ("noise", 20, False), ("noise", 30, False),
                ("scale", 0.75, True), ("scale", 0.5, True), ("scale", 0.35, False),
                ("jpeg", 30, True), ("jpeg", 10, False)]


def gt_table(records, thr):
    rows = []
    for r in records:
        m = match_image(r["preds"], r["gts"], thr)
        for j, g in enumerate(r["gts"]):
            w = g["box"][2] - g["box"][0]
            rows.append({"cls": g["cls"], "hit": m["gt_hit"][j], "size": band_size(w), "bright": band_brightness(r["brightness"]),
                         "blur": band_blur(r["blur"]), "source": SOURCE_NAMES.get(source_of(r["path"]), "?"), "path": r["path"], "w": w})
    return pd.DataFrame(rows)


def build_pack(detector, name, val_records, test_records, thr, with_robustness=True):
    pic = per_image_counts(test_records, thr)
    m = metrics_from(pic)
    no_gt = np.array([len(r["gts"]) == 0 for r in test_records])
    ap, mAP = dataset_ap50(test_records)
    g = gt_table(test_records, thr)
    other = [m["cls_recall"][c] for c in range(NC) if c != STOP and m["cls_n_gt"][c] > 0]
    pack = {"name": name, "detector": detector, "thr": thr, "val": val_records, "test": test_records,
            "pic": pic, "metrics": m, "ap": ap, "map50": mAP, "gt": g,
            "req": {"stop_recall": m["cls_recall"][STOP], "other_min_recall": min(other) if other else float("nan"),
                    "bg_fa_per_100": pic[no_gt][:, :, 1].sum() / max(no_gt.sum(), 1) * 100,
                    "worst_odd_f1": None, "p95_latency_ms": None, "distance_err_pct": None}}
    sl = {}    # slices: name -> dict(value, n, metric, ci)
    for key, label in [("cls", "class"), ("source", "source"), ("size", "size"), ("bright", "brightness"), ("blur", "blur")]:
        for v, sub in g.groupby(key):
            sl[f"{label}={TARGET_CLASSES[v] if key == 'cls' else v}"] = {"value": sub.hit.mean(), "n": len(sub), "metric": "recall",
                                                                         "ci": wilson(int(sub.hit.sum()), len(sub))}
    if with_robustness:
        rob_rows = []
        for kind, level, in_odd in DEGRADATIONS:
            recs = run_detector(detector, ITEMS["test"], transform=lambda im, k=kind, l=level: degrade(im, k, l))
            mm = metrics_from(per_image_counts(recs, thr))
            f1 = nan0(mm["f1"])
            rob_rows.append({"kind": kind, "level": level, "in_odd": in_odd, "f1": f1, "recall": mm["recall"], "precision": mm["precision"]})
            sl[f"degrade:{kind}={level}"] = {"value": f1, "n": len(recs), "metric": "F1", "ci": (float("nan"), float("nan"))}
        pack["robust"] = rob = pd.DataFrame(rob_rows)
        pack["req"]["worst_odd_f1"] = float(rob[rob.in_odd].f1.min())
    pack["slices"] = sl
    return pack


t0 = time.time()
PACK = {"candidate": build_pack(DET_MAIN, "candidate", VAL_MAIN, TEST_MAIN, OP_THR)}
if DET_V1:
    PACK["v1"] = build_pack(DET_V1, "v1", run_detector(DET_V1, ITEMS["val"]), run_detector(DET_V1, ITEMS["test"]), OP_THR)
print(f"evaluated in {time.time() - t0:.0f}s at threshold {OP_THR}")

In [ ]:
P = PACK["candidate"]; M = P["metrics"]
print(f"TEST @ thr {OP_THR}:  precision {M['precision']:.3f} | recall {M['recall']:.3f} | F1 {M['f1']:.3f} | mAP@0.5 (own) {P['map50']:.3f}")
lo, hi = np.nanpercentile([metrics_from(P["pic"][np.random.RandomState(i).randint(0, len(P["pic"]), len(P["pic"]))])["f1"] for i in range(400)], [2.5, 97.5])
print(f"F1 95% bootstrap interval over test images: [{lo:.2f}, {hi:.2f}]  <- the test set is small, so the average alone is not trustworthy\n")
display(pd.DataFrame({"n_truth": M["cls_n_gt"], "precision": np.round(M["cls_precision"], 3), "recall": np.round(M["cls_recall"], 3),
                      "AP@0.5": [round(P["ap"].get(c, float("nan")), 3) for c in range(NC)]}, index=TARGET_CLASSES))

### Slices — where does it break?
Bars show **recall**; whiskers are 95% intervals. Slices with few samples (n < 20) are flagged: you cannot conclude anything from them.
Find the weakest slice. Is it weak because the model is bad there, or because there are only a handful of examples?

In [ ]:
sl = {k: v for k, v in P["slices"].items() if v["metric"] == "recall"}
vals, ci = np.array([v["value"] for v in sl.values()]), np.array([v["ci"] for v in sl.values()])
err = np.clip([vals - ci[:, 0], ci[:, 1] - vals], 0, None)   # clip float rounding at recall = 1.0
plt.figure(figsize=(11, 3.6))
plt.bar(range(len(sl)), vals, yerr=err, color=["tab:red" if v["n"] < 20 else "tab:blue" for v in sl.values()], capsize=3)
plt.xticks(range(len(sl)), [f"{k}\n(n={v['n']})" for k, v in sl.items()], rotation=60, ha="right", fontsize=7)
plt.axhline(REQS[0].target, color="k", ls=":", label="R1 target (Stop)"); plt.ylim(0, 1.05); plt.ylabel("recall"); plt.legend()
plt.title("Test recall by slice (red = fewer than 20 samples)"); finish_plot("slices")

### Robustness under degradations
Each test image is degraded on purpose (darker, blurrier, noisier, smaller, more compressed) and evaluated again, for both the candidate and v1.
Only conditions **inside the ODD** count for requirement R5; the others show where the system stops being usable (and what monitoring in Step 5 and 6 must detect).

In [ ]:
fig, ax = plt.subplots(1, 5, figsize=(16, 3))
for a, (kind, sub) in zip(ax, P["robust"].groupby("kind", sort=False)):
    a.bar([str(l) for l in sub.level], sub.f1, color=["tab:blue" if o else "lightgray" for o in sub.in_odd])
    a.axhline(M["f1"], color="k", ls=":"); a.axhline(REQS[4].target, color="r", ls="--"); a.set_ylim(0, 1); a.set_title(kind)
ax[0].set_ylabel("F1")
plt.suptitle("F1 under degradation (grey = outside the ODD, dotted = clean F1, red = R5 limit)", y=1.02, fontsize=9); finish_plot("robustness")
if "v1" in PACK:
    display(pd.DataFrame({k: PACK[k]["robust"].set_index(["kind", "level"]).f1 for k in PACK}).round(3))

## 3.5 Failure mining
Every miss and every false alarm gets a **reason**. The histogram of reasons is the to-do list for the next iteration:
*small sign* → more resolution / more small examples, *dark* → night data, *wrong class* → look at 30 vs 50, and so on.

**What to check:** look at the false alarms on background images in the grid. Are they all real model errors, or do some show a real sign that nobody labelled? (Section 2.6 found many of those.)

In [ ]:
def mine_failures(records, thr):
    out = []
    for r in records:
        m = match_image(r["preds"], r["gts"], thr)
        wrong_gt = {j: i for i, j in m["wrong"].items()}
        kept = [r["preds"][i] for i in m["keep"]]
        for j, hit in enumerate(m["gt_hit"]):
            if hit:
                continue
            g = r["gts"][j]
            near_same = [p for p in r["preds"] if p["cls"] == g["cls"] and p["conf"] >= thr * 0.5 and 0.1 <= iou_xyxy(p["box"], g["box"]) < 0.5]
            if j in wrong_gt:
                reason = f"wrong class (read as {TARGET_CLASSES[r['preds'][wrong_gt[j]]['cls']]})"
            elif near_same:
                reason = "poor localisation"
            elif g["box"][2] - g["box"][0] < SIZE_EDGES[0]:
                reason = "small sign"
            elif band_brightness(r["brightness"]) == "dark":
                reason = "dark scene"
            elif band_blur(r["blur"]) == "blurry":
                reason = "blurry image"
            else:
                reason = "unexplained miss"
            out.append({"type": "MISS", "reason": reason, "path": r["path"], "gt": g, "preds": kept})
        out += [{"type": "FALSE ALARM", "reason": "on background image" if not r["gts"] else "extra / duplicate near a sign",
                 "path": r["path"], "gt": r["gts"][0] if r["gts"] else None, "preds": [r["preds"][i]]}
                for i in m["keep"] if not (m["pred_tp"][i] or i in m["wrong"])]
    return out


FAILS = mine_failures(TEST_MAIN, OP_THR)
cnt = Counter((f["type"], f["reason"].split(" (")[0]) for f in FAILS)
fdf = pd.DataFrame([{"type": t, "reason": r, "count": c} for (t, r), c in cnt.items()]).sort_values("count", ascending=False)
display(fdf.set_index(["type", "reason"]))
vis = []
for f in FAILS[:12]:
    im = cv2.imread(f["path"])
    if f["gt"] is not None:
        im = draw_boxes(im, [{"box": f["gt"]["box"], "label": "truth: " + TARGET_CLASSES[f["gt"]["cls"]]}], (0, 200, 0))
    vis.append(draw_boxes(im, [{"box": p["box"], "label": f"{TARGET_CLASSES[p['cls']]} {p['conf']:.2f}"} for p in f["preds"]], (0, 0, 255), 1))
show_grid(vis, [f"{f['type']}: {f['reason']}" for f in FAILS[:12]], cols=4, size=3.2)

## 3.6 The release gate
Inputs: the **requirements** (Step 1) and the **currently deployed model**. Output: SHIP / NO-SHIP / HOLD.
* Every requirement must **PASS** (not measured yet → **PENDING**).
* No slice may get worse than the deployed model by more than 5 points (if a slice has fewer than 20 samples → **REVIEW**, a human must look).

At this point latency (R4) and distance error (R6) are not measured yet, so they show as PENDING and the gate cannot say SHIP. That is the gate doing its job:
no evidence, no release. R4 is measured in Step 4 and the gate is re-run there.

In [ ]:
def req_rows(req):
    """-> (requirement, measured value rounded for display, check result True/False/None) for every requirement"""
    for r in REQS:
        v = req.get(r.metric)
        yield r, None if v is None else round(float(v), 3), r.check(v)


def release_gate(cand, base=None, max_regress=0.05, min_n=20, verbose=True):
    rows = [{"gate": r.rid, "what": r.text[:58], "value": v, "limit": f"{r.op} {r.target}{r.unit}",
             "status": "PENDING" if ok is None else ("PASS" if ok else "FAIL")} for r, v, ok in req_rows(cand["req"])]
    if base is not None:
        for s, c in cand["slices"].items():
            b = base["slices"].get(s)
            if b is None or c["value"] != c["value"] or b["value"] != b["value"]:
                continue
            if b["value"] - c["value"] > max_regress:
                rows.append({"gate": "regression", "what": s, "value": round(c["value"], 3), "limit": f"deployed {b['value']:.3f} -0.05",
                             "status": "FAIL" if min(c["n"], b["n"]) >= min_n else "REVIEW"})
    df = pd.DataFrame(rows)
    st = set(df.status)
    decision = "NO-SHIP" if "FAIL" in st else ("HOLD (evidence missing / needs review)" if ({"PENDING", "REVIEW"} & st) else "SHIP")
    if verbose:
        display(df.set_index("gate"))
        print("DECISION:", decision)
    return decision, df


GATE_DECISION, GATE_TABLE = release_gate(PACK["candidate"], PACK.get("v1"))

**Exercise.** (1) Which single failing or pending line would you work on first, and how?
(2) The cell below calls `release_gate(PACK['v1'], PACK['candidate'])`: it pretends the *old* model is the new candidate and the candidate is the one in the car. Read the output: this is what a regression looks like, and why the gate refuses it.
(3) Would you change `max_regress` or `min_n`? What does each choice cost you?

In [ ]:
# YOUR TURN: a candidate that is "better on average" but worse where it matters
_ = release_gate(PACK["v1"], PACK["candidate"]) if "v1" in PACK else print("train the v1 baseline to run this exercise")

**Check yourself (Step 3)**
1. Why must the threshold be chosen on validation and not on test? What would happen to the test score if you did it on test?
2. The candidate has a higher average than v1, but the gate would still block it if one slice got 6 points worse. Is that too strict? Give an example where it saves you.
3. A slice shows recall 1.00 with n = 4. What can you conclude?
4. The requirements file says R3 was relaxed from 2.0 to 4.0 false alarms per 100 background images. When is it acceptable to change a requirement after seeing results, and what should be written down when you do?

# Step 4 — Deployment constraints
**What you will learn:** how to make a model fast enough for the car without silently breaking it.

A model that is accurate but misses its **latency budget** is a failed model. Real-time means a *guaranteed worst case*, not a good average,
and every optimisation must be **re-validated** — quantisation can quietly hurt the small and rare cases.

**Background.** In a car the model runs on an embedded chip (an SoC or a small edge GPU) with strict power, heat and memory limits.
The usual toolbox: a smaller model or input size, half precision (FP16), 8-bit integers (INT8), pruning, and inference compilers (ONNX Runtime, TensorRT, OpenVINO).
Here R4 asks for p95 latency ≤ 66 ms, i.e. at least 15 frames per second for 95% of frames.

The plan for this step: measure latency properly → export variants → evaluate each variant with the **same** code and threshold → pick the one that passes R1 *and* R4.
Note: your timings depend on your GPU. A Colab T4 is slower than a recent laptop GPU, and the car's chip would be slower again.

## 4.1 Measuring latency properly
Latency is reported as **p50 / p95 / p99 / max**, not just the mean, on the full pipeline (pre-processing + inference + post-processing).
p95 = 40 ms means 95% of frames finish within 40 ms; the remaining 5% are the ones that can make the car react late.

In [ ]:
BENCH_IMGS = [cv2.imread(it["path"]) for it in ITEMS["test"][:40]]


def benchmark(fn, n=150, warmup=20):
    for i in range(warmup):
        fn(BENCH_IMGS[i % len(BENCH_IMGS)])
    ts = []
    for i in range(n):
        im = BENCH_IMGS[i % len(BENCH_IMGS)]
        t0 = time.perf_counter(); fn(im); ts.append((time.perf_counter() - t0) * 1000)
    p50, p95, p99 = np.percentile(ts, [50, 95, 99])
    return {"p50_ms": p50, "p95_ms": p95, "p99_ms": p99, "max_ms": np.max(ts), "fps_p50": 1000 / p50}

## 4.2 A tiny ONNX inference path (letterbox → ORT → decode → NMS)
Written by hand so exported models go through the same evaluation pipeline, and so you can see exactly what an embedded runtime has to do around the model itself.

In [ ]:
import onnxruntime as ort


def letterbox(img, size, color=114):
    h, w = img.shape[:2]
    s = min(size / h, size / w)
    nh, nw = int(round(h * s)), int(round(w * s))
    top, left = (size - nh) // 2, (size - nw) // 2
    canvas = np.full((size, size, 3), color, np.uint8)
    canvas[top:top + nh, left:left + nw] = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)
    return canvas, s, left, top


def to_input(lb):
    """letterboxed BGR uint8 image -> contiguous 1x3xHxW float32 RGB tensor in [0, 1]"""
    return np.ascontiguousarray(lb[:, :, ::-1].transpose(2, 0, 1)[None].astype(np.float32) / 255.0)


def _area(b):
    return (b[..., 2] - b[..., 0]) * (b[..., 3] - b[..., 1])


def nms_per_class(boxes, scores, cls, iou_thr):
    """Greedy non-maximum suppression, applied separately for each class. Returns indices to keep."""
    order, keep = np.argsort(-scores), []
    while len(order):
        i, rest = order[0], order[1:]
        keep.append(int(i))
        inter = (np.clip(np.minimum(boxes[i, 2], boxes[rest, 2]) - np.maximum(boxes[i, 0], boxes[rest, 0]), 0, None)
                 * np.clip(np.minimum(boxes[i, 3], boxes[rest, 3]) - np.maximum(boxes[i, 1], boxes[rest, 1]), 0, None))
        iou_ = inter / np.maximum(_area(boxes[i]) + _area(boxes[rest]) - inter, 1e-9)
        order = rest[~((iou_ > iou_thr) & (cls[rest] == cls[i]))]
    return keep


class OrtDetector:
    def __init__(self, onnx_path, imgsz=IMGSZ, providers=None, label=None):
        providers = providers or (["CUDAExecutionProvider"] if HAS_GPU else []) + ["CPUExecutionProvider"]
        self.sess = ort.InferenceSession(str(onnx_path), providers=providers)
        self.inp, self.imgsz, self.label = self.sess.get_inputs()[0].name, imgsz, label or Path(str(onnx_path)).stem
        self.provider = self.sess.get_providers()[0]
        self(np.zeros((416, 416, 3), np.uint8))

    def __call__(self, img, conf=0.01, iou=0.7):
        lb, s, left, top = letterbox(img, self.imgsz)
        out = self.sess.run(None, {self.inp: to_input(lb)})[0][0].T          # (anchors, 4 + nc)
        cls = out[:, 4:].argmax(1); confs = out[np.arange(len(out)), 4 + cls]
        sel = confs >= conf
        out, cls, confs = out[sel], cls[sel], confs[sel]
        if len(out) == 0:
            return []
        cx, cy, w, h = out[:, :4].T
        boxes = np.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], 1)               # xyxy in letterbox pixels
        order = np.argsort(-confs)[:1000]
        boxes, cls, confs = boxes[order], cls[order], confs[order]
        H, W = img.shape[:2]
        off, lim = (left, top), (W, H)                    # undo the letterbox, clip to the image
        return [{"box": tuple(float(np.clip((boxes[i, k] - off[k % 2]) / s, 0, lim[k % 2])) for k in range(4)),
                 "cls": int(cls[i]), "conf": float(confs[i])} for i in nms_per_class(boxes, confs, cls, iou)]

## 4.3 Build the variants
| Variant | Why it exists |
|---|---|
| PyTorch FP32 @640 | reference (what we validated) |
| PyTorch FP32 @416 | cheaper input — but small signs have fewer pixels |
| PyTorch FP16 @640 | half precision on GPU, usually free accuracy |
| PyTorch FP32 on CPU | what a laptop / small board without GPU would do |
| ONNX FP32 | portable graph for ONNX Runtime / TensorRT / OpenVINO toolchains |
| ONNX INT8 | 4x smaller weights, fast on CPU/NPUs — **the risky one** |
| TensorRT FP16 (optional) | vendor-optimised engine for NVIDIA edge devices |

In [ ]:
VARIANTS = {"PyTorch FP32 @640": DET_MAIN, "PyTorch FP32 @416": YoloDetector(WEIGHTS_MAIN, 416, label="fp32_416")}
if HAS_GPU:
    VARIANTS["PyTorch FP16 @640"] = YoloDetector(WEIGHTS_MAIN, IMGSZ, half=True, label="fp16")
VARIANTS["PyTorch FP32 CPU @640"] = YoloDetector(WEIGHTS_MAIN, IMGSZ, device="cpu", label="cpu")

onnx_path = None
try:
    onnx_path = Path(YOLO(str(WEIGHTS_MAIN)).export(format="onnx", imgsz=IMGSZ, opset=12, simplify=True, dynamic=False, device="cpu"))
    VARIANTS["ONNX FP32"] = OrtDetector(onnx_path, IMGSZ, label="onnx_fp32")
    print("ONNX exported:", onnx_path, f"({onnx_path.stat().st_size / 1e6:.1f} MB) | provider:", VARIANTS["ONNX FP32"].provider)
except Exception as e:
    print("ONNX export failed:", e)

In [ ]:
# INT8 post-training static quantisation (needs a small calibration set of real images)
# per_channel=False: per-channel bias quantisation produces an invalid graph with onnx 1.23 + onnxruntime 1.31
if onnx_path is not None:
    try:
        import onnx
        from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantFormat, QuantType

        class CalibReader(CalibrationDataReader):
            def __init__(self, paths, name, size):
                self.it, self.name, self.size = iter(paths), name, size

            def get_next(self):
                p = next(self.it, None)
                return None if p is None else {self.name: to_input(letterbox(cv2.imread(p), self.size)[0])}

        calib = [it["path"] for it in random.Random(SEED).sample(ITEMS["train"], 64)]
        in_name = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"]).get_inputs()[0].name
        head_idx = len(YOLO(str(WEIGHTS_MAIN)).model.model) - 1      # last module = detection head (22 in YOLOv8n, 23 in YOLO11)
        head = [n.name for n in onnx.load(str(onnx_path)).graph.node if n.name.startswith(f"/model.{head_idx}/")]   # keep the head in float
        int8_path = onnx_path.with_name(onnx_path.stem + "_int8.onnx")
        quantize_static(str(onnx_path), str(int8_path), CalibReader(calib, in_name, IMGSZ), quant_format=QuantFormat.QDQ,
                        per_channel=False, weight_type=QuantType.QInt8, activation_type=QuantType.QInt8, nodes_to_exclude=head)
        VARIANTS["ONNX INT8 (CPU)"] = OrtDetector(int8_path, IMGSZ, providers=["CPUExecutionProvider"], label="onnx_int8")
        print(f"INT8 model: {int8_path.stat().st_size / 1e6:.1f} MB (FP32: {onnx_path.stat().st_size / 1e6:.1f} MB)")
    except Exception as e:
        print("INT8 quantisation skipped:", repr(e)[:300])

In [ ]:
if TRY_TENSORRT and HAS_GPU:
    try:
        pip_install("tensorrt")
        eng = Path(YOLO(str(WEIGHTS_MAIN)).export(format="engine", half=True, imgsz=IMGSZ, device=0))
        VARIANTS["TensorRT FP16"] = YoloDetector(eng, IMGSZ, label="trt")
    except Exception as e:
        print("TensorRT skipped:", repr(e)[:300])

## 4.4 Benchmark **and** re-validate every variant
Same test images, same operating threshold, same metric code. If a variant is fast but loses Stop recall, it is not an optimisation — it is a regression.

**What to check:** compare INT8 with FP32. How much faster is it on your hardware, and what did it cost in Stop recall and F1? Which variant would you put in the car?

In [ ]:
rows, VARIANT_PACKS = [], {}
for name, det in VARIANTS.items():
    lat = benchmark(lambda im, d=det: d(im, OP_THR), n=40 if "CPU" in name or "INT8" in name else 150)   # realistic: run at the operating threshold
    pk = VARIANT_PACKS[name] = build_pack(det, name, [], run_detector(det, ITEMS["test"]), OP_THR, with_robustness=False)
    rows.append({"variant": name, **{k: round(v, 1) for k, v in lat.items()}, "F1": round(pk["metrics"]["f1"], 3),
                 "Stop recall": round(pk["req"]["stop_recall"], 3), "mAP50": round(pk["map50"], 3)})
bench = pd.DataFrame(rows).set_index("variant")
R1, R4 = REQS[0], REQS[3]
bench["R1 recall"] = ["PASS" if R1.check(v) else "FAIL" for v in bench["Stop recall"]]
bench["R4 latency"] = ["PASS" if R4.check(v) else "FAIL" for v in bench["p95_ms"]]
display(bench)

In [ ]:
ax = bench[["p50_ms", "p95_ms", "p99_ms"]].plot(kind="barh", figsize=(9, 3.6), width=0.8)
ax.axvline(R4.target, color="r", ls="--"); ax.set_xlabel("latency (ms), red line = R4 budget"); ax.set_title("Latency percentiles per variant")
finish_plot("latency")

## 4.5 Pick the deployable variant and re-run the gate
Rule: among variants that **pass R1 and R4**, take the most accurate. If none passes, the variant closest to passing (best Stop recall) is carried forward and the gate stays closed;
the next discussion is then about hardware, input size or training, not about shipping.
The gate is re-run with the deployable variant, now with a measured R4. Only R6 (simulation, see section 6.4) should still be pending.

In [ ]:
ok = bench[(bench["R1 recall"] == "PASS") & (bench["R4 latency"] == "PASS")]
if len(ok):
    DEPLOY_NAME = ok.sort_values(["F1", "p95_ms"], ascending=[False, True]).index[0]
    print("Deployable variant:", DEPLOY_NAME)
else:
    DEPLOY_NAME = bench.sort_values(["Stop recall", "F1", "p95_ms"], ascending=[False, False, True]).index[0]
    print("No variant passes both R1 and R4. Closest to passing:", DEPLOY_NAME, "-> the gate stays closed until the failing line is fixed.")

PACK["candidate"]["req"]["p95_latency_ms"] = float(bench.loc[DEPLOY_NAME, "p95_ms"])
if DEPLOY_NAME != "PyTorch FP32 @640":      # the shipped artifact is the optimised one: its accuracy numbers count
    for k in ("stop_recall", "other_min_recall", "bg_fa_per_100"):
        PACK["candidate"]["req"][k] = VARIANT_PACKS[DEPLOY_NAME]["req"][k]
PACK["candidate"]["deploy_variant"] = DEPLOY_NAME
GATE_DECISION, GATE_TABLE = release_gate(PACK["candidate"], PACK.get("v1"))

**Check yourself (Step 4)**
1. Two models have the same mean latency of 30 ms. One has p99 = 35 ms, the other p99 = 120 ms. Which one goes in the car, and why?
2. Why is re-validating after INT8 quantisation not optional? Which classes or slices would you expect to suffer first?
3. Reducing the input from 640 to 416 pixels is faster. Which requirement is most at risk, and which slice from Step 3 shows it?

# Step 5 — The perception stack
**What you will learn:** what has to be built around a detector before its output can drive a car.

A detector is one box in a longer chain. Production systems add **temporal filtering** (tracking), **geometry** (calibrated distance), **cross-checks** (maps, other sensors) and **fallbacks** (what to do when perception is unreliable).

`camera -> detector -> tracker -> distance -> plausibility check -> health monitor -> decision`

The four ideas in this step:
* one detection is not one sign: detections flicker from frame to frame, so they are tracked over time;
* distance needs a **calibrated** camera, not a guessed field of view;
* a map prior catches readings that make no sense;
* the system must *know* when it cannot see, and fall back to something safer.

## 5.1 Distance from a single camera — and how wrong it can be
Pinhole model: **Z = f · H / h** (focal length in pixels × real sign height ÷ height in pixels).
Three error sources: a wrong assumed **sign height**, a wrong **focal length** (field of view guessed instead of calibrated), and **±2 px** of box error — which explodes with distance because far signs are only a few pixels tall.

**What to check:** in the plot, at what distance does the ±2 px error alone break the 15% limit of R6?

In [ ]:
@dataclass
class Camera:
    width: int
    height: int
    fx: float
    fy: float
    cx: float
    cy: float

    @classmethod
    def from_hfov(cls, width, height, hfov_deg):
        f = (width / 2) / math.tan(math.radians(hfov_deg) / 2)
        return cls(width, height, f, f, width / 2, height / 2)

    def distance(self, h_px, real_h):
        return self.fy * real_h / max(h_px, 1e-6)

    def height_px(self, dist, real_h):
        return self.fy * real_h / dist


TRUE_CAM = Camera(640, 480, 800.0, 800.0, 320.0, 240.0)           # the real camera (hfov ~ 43.6 deg)
ASSUMED = Camera.from_hfov(640, 480, 60)                          # a typical "guess"
REAL_H, ASSUMED_H = 0.60, 0.75
Z = np.linspace(5, 50, 100)
h_true = TRUE_CAM.height_px(Z, REAL_H)
plt.figure(figsize=(7, 3.4))
plt.plot(Z, np.full_like(Z, (ASSUMED_H / REAL_H - 1) * 100), label=f"sign height assumed {ASSUMED_H} m instead of {REAL_H} m")
plt.plot(Z, np.full_like(Z, (ASSUMED.fy / TRUE_CAM.fy - 1) * 100), label="focal length from guessed 60 deg FOV (true 43.6 deg)")
plt.plot(Z, (TRUE_CAM.fy * REAL_H / np.maximum(h_true - 2, 0.5) / Z - 1) * 100, label="box 2 px too small")
for sign in (1, -1):
    plt.axhline(sign * REQS[5].target, color="k", ls=":")
plt.xlabel("true distance (m)"); plt.ylabel("distance error (%)"); plt.legend(fontsize=7); plt.title("Where distance errors come from (dotted = R6 limit)")
finish_plot("distance_sensitivity")

## 5.2 Calibration fixes the focal-length error
The cell simulates photographing a chessboard from 14 poses with a camera of known focal length (800 px), adds 0.3 px of corner noise, and lets OpenCV recover it.
On a real car you do exactly this once per camera, with a printed board. Compare the recovered focal length with the true one.

In [ ]:
def simulate_calibration(K_true, n_views=14, noise=0.3, seed=SEED, size=(640, 480)):
    rng = np.random.RandomState(seed)
    cols, rows, sq = 9, 6, 0.025
    objp = np.zeros((rows * cols, 3), np.float32)
    objp[:, :2] = np.mgrid[0:cols, 0:rows].T.reshape(-1, 2) * sq
    objpts, imgpts = [], []
    while len(objpts) < n_views:
        rvec = rng.uniform(-0.5, 0.5, 3)
        tvec = np.array([rng.uniform(-0.15, -0.05), rng.uniform(-0.1, -0.02), rng.uniform(0.5, 0.9)])
        p = cv2.projectPoints(objp, rvec, tvec, K_true, np.zeros(5))[0].reshape(-1, 2)
        if ((p < 5) | (p > np.array(size) - 5)).any():          # board must be fully inside the image
            continue
        objpts.append(objp); imgpts.append((p + rng.normal(0, noise, p.shape)).astype(np.float32).reshape(-1, 1, 2))
    rms, K, *_ = cv2.calibrateCamera(objpts, imgpts, size, None, None)
    return rms, K


K_true = np.array([[TRUE_CAM.fx, 0, TRUE_CAM.cx], [0, TRUE_CAM.fy, TRUE_CAM.cy], [0, 0, 1.0]])
rms, K_est = simulate_calibration(K_true)
CALIB_CAM = Camera(640, 480, float(K_est[0, 0]), float(K_est[1, 1]), float(K_est[0, 2]), float(K_est[1, 2]))
print(f"reprojection RMS = {rms:.2f} px | focal estimate fx = {CALIB_CAM.fx:.1f} px (true 800.0, error {abs(CALIB_CAM.fx / 800 - 1) * 100:.1f}%)")
z_true, h_px = 20.0, TRUE_CAM.height_px(20.0, REAL_H)
for nm, cam in [("guessed 60 deg FOV", ASSUMED), ("calibrated", CALIB_CAM)]:
    d = cam.distance(h_px, REAL_H)
    print(f"  {nm:20s}: sign really at {z_true:.0f} m -> estimated {d:.1f} m  ({(d / z_true - 1) * 100:+.1f}%)")

## 5.3 Tracking — one sign, one identity
A raw detector flickers: a sign is found in frame 1, lost in frame 2, found again in frame 3. A **Kalman filter** predicts where the box should be, **Hungarian matching** links detections to tracks,
a track is *confirmed* only after 3 hits, and the class is decided by a **confidence-weighted vote** over the last 10 frames (so one wrong frame cannot flip "Speed 30" into "Speed 50").

In [ ]:
def box_to_z(b):
    return np.array([(b[0] + b[2]) / 2, (b[1] + b[3]) / 2, b[2] - b[0], b[3] - b[1]], float)


def z_to_box(z):
    cx, cy, w, h = z[:4]
    return (cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2)


class KalmanBox:
    """Constant-velocity Kalman filter on [cx, cy, w, h, vx, vy, vw, vh]."""
    def __init__(self, box, meas_std=2.0):
        self.x = np.concatenate([box_to_z(box), np.zeros(4)])
        self.P = np.diag([10.0] * 4 + [100.0] * 4)
        self.F = np.eye(8); self.F[:4, 4:] = np.eye(4)
        self.H = np.eye(4, 8)
        self.Q = np.diag([1.0] * 4 + [0.5] * 4) * 0.1
        self.R = np.eye(4) * meas_std ** 2

    def predict(self):
        self.x = self.F @ self.x
        self.P = self.F @ self.P @ self.F.T + self.Q
        return z_to_box(self.x)

    def update(self, box):
        y = box_to_z(box) - self.H @ self.x
        K = self.P @ self.H.T @ np.linalg.inv(self.H @ self.P @ self.H.T + self.R)
        self.x = self.x + K @ y
        self.P = (np.eye(8) - K @ self.H) @ self.P


class Track:
    _next = 0

    def __init__(self, det):
        Track._next += 1
        self.id = Track._next
        self.kf, self.box = KalmanBox(det["box"]), det["box"]
        self.votes = deque([(det["cls"], det["conf"])], maxlen=10)
        self.hits, self.misses, self.dist = 1, 0, None

    def predict(self):
        self.box = self.kf.predict()
        return self.box

    def update(self, det):
        self.kf.update(det["box"]); self.box = z_to_box(self.kf.x)
        self.votes.append((det["cls"], det["conf"])); self.hits += 1; self.misses = 0

    @property
    def label(self):
        w = Counter()
        for c, s in self.votes:
            w[c] += s
        cls = w.most_common(1)[0][0]
        return cls, float(np.mean([s for c, s in self.votes if c == cls]))


class SignTracker:
    def __init__(self, iou_gate=0.1, max_age=6, min_hits=3):
        self.tracks, self.iou_gate, self.max_age, self.min_hits = [], iou_gate, max_age, min_hits

    def step(self, dets):
        preds = [t.predict() for t in self.tracks]
        matched_t, matched_d = set(), set()
        if self.tracks and dets:
            cost = np.array([[1 - iou_xyxy(p, d["box"]) for d in dets] for p in preds])
            for r, c in zip(*linear_sum_assignment(cost)):
                if 1 - cost[r, c] >= self.iou_gate:
                    self.tracks[r].update(dets[c]); matched_t.add(r); matched_d.add(c)
        for i, t in enumerate(self.tracks):
            if i not in matched_t:
                t.misses += 1
        self.tracks += [Track(d) for j, d in enumerate(dets) if j not in matched_d]
        self.tracks = [t for t in self.tracks if t.misses <= self.max_age]
        return [t for t in self.tracks if t.hits >= self.min_hits and t.misses <= 1]


# tiny check: a detector that drops 40% of frames and sometimes mislabels -> tracker output stays stable
rng = np.random.RandomState(1)
tk, trace = SignTracker(), []
for f in range(40):
    box = (200 + 3 * f, 150 + f, 240 + 4 * f, 190 + 4 * f)
    dets = [] if rng.rand() < 0.4 else [{"box": tuple(b + rng.normal(0, 1.5) for b in box), "cls": 2 if rng.rand() < 0.15 else 1, "conf": 0.6 + 0.3 * rng.rand()}]
    out = tk.step(dets)
    trace.append((len(dets), len(out), TARGET_CLASSES[out[0].label[0]] if out else "-"))
print("frames with a raw detection:", sum(t[0] for t in trace), "/ 40  | frames with a stable track:", sum(t[1] for t in trace), "/ 40")
print("class shown by the track:", Counter(t[2] for t in trace))

## 5.4 Plausibility check against a map prior
The camera says "Speed Limit 30", but the map says this road is 50 and a 120 sign in a 30 zone makes no sense. Production systems **fuse** vision with other sources and flag disagreement instead of blindly obeying the camera.

In [ ]:
def speed_of(cls_name):
    return int(cls_name.split()[-1]) if cls_name.startswith("Speed Limit") else None


def plausibility(detected_kmh, map_kmh, max_step=30):
    if map_kmh is None:
        return "NO_MAP"
    if detected_kmh == map_kmh:
        return "CONSISTENT"
    return "ZONE_CHANGE_PLAUSIBLE" if abs(detected_kmh - map_kmh) <= max_step else "SUSPECT"


pd.DataFrame([(d, m, plausibility(d, m)) for d, m in [(50, 50), (30, 50), (120, 30), (50, None)]],
             columns=["sign reads (km/h)", "map says (km/h)", "verdict"])

## 5.5 Knowing when you cannot see: a health monitor with fallback states
If the image is too dark or too blurry, detections cannot be trusted — *and the absence of detections is not "no signs"*. A small state machine tracks image health with hysteresis (no flapping):

| State | Meaning | Action |
|---|---|---|
| NOMINAL | image quality inside the ODD | use camera signs |
| DEGRADED | dark / blurry | lower trust, prefer map speed limit |
| FAILSAFE | very dark for a long time | ignore camera, alert the driver |

In [ ]:
class PerceptionHealth:
    ACTIONS = {"NOMINAL": "use camera signs", "DEGRADED": "prefer map limit, lower trust", "FAILSAFE": "ignore camera, alert driver"}

    def __init__(self, dark_thr, blur_thr, fail_thr, to_degraded=3, to_failsafe=8, to_nominal=10):
        self.dark_thr, self.blur_thr, self.fail_thr = dark_thr, blur_thr, fail_thr
        self.cfg = (to_degraded, to_failsafe, to_nominal)
        self.state, self.bad, self.good, self.very_bad = "NOMINAL", 0, 0, 0

    def update(self, brightness, blur):
        bad = brightness < self.dark_thr or blur < self.blur_thr
        self.bad, self.good = (self.bad + 1, 0) if bad else (0, self.good + 1)
        self.very_bad = self.very_bad + 1 if brightness < self.fail_thr else 0
        d, f, n = self.cfg
        if self.state == "NOMINAL" and self.bad >= d:
            self.state = "DEGRADED"
        elif self.state == "DEGRADED" and self.very_bad >= f:
            self.state = "FAILSAFE"
        elif self.state in ("DEGRADED", "FAILSAFE") and self.good >= n:
            self.state = "NOMINAL"
        return self.state

## 5.6 Put it together on a video
The demo video **simulates a drive**: it zooms into 8 test images (the car approaches the sign) and fades the light progressively in the second half (dusk → night).
Watch the *confirmed* tracks, the distance, and the health monitor moving from NOMINAL to DEGRADED and FAILSAFE. The detection log (`detections_log.csv`) records every frame.
*(This is a synthetic stand-in: real footage gives more convincing results, and the simulation in section 6.4 gives distance **ground truth**.)*

In [ ]:
def build_demo_video(path, n_clips=8, frames=24, size=640, fps=20, fade_start=0.5, min_brightness=0.12):
    pool = [it for it in ITEMS["test"] if it["gts"]]
    clips = random.Random(SEED).sample(pool, min(n_clips, len(pool)))
    total = len(clips) * frames
    wr = cv2.VideoWriter(str(path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (size, size))
    n = 0
    for it in clips:
        img = cv2.imread(it["path"]); H, W = img.shape[:2]
        b = it["gts"][0]["box"]; cx, cy = (b[0] + b[2]) / 2, (b[1] + b[3]) / 2
        for k in range(frames):                                   # slow zoom towards the sign
            s = 1 + 0.8 * k / (frames - 1)
            cw, ch = W / s, H / s
            x0, y0 = int(np.clip(cx - cw / 2, 0, W - cw)), int(np.clip(cy - ch / 2, 0, H - ch))
            fr = cv2.resize(img[y0:y0 + int(ch), x0:x0 + int(cw)], (size, size))
            p = n / total
            if p > fade_start:                                    # second half: fade to dark
                fr = degrade(fr, "brightness", 1 - (1 - min_brightness) * (p - fade_start) / (1 - fade_start))
            wr.write(fr); n += 1
    wr.release()
    return Path(path)


def run_pipeline(video, detector, thr, camera, out_path, csv_path, sign_h=0.60, map_limit=50, warn_dist=10.0):
    cap = cv2.VideoCapture(str(video))
    W, H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps_in = cap.get(cv2.CAP_PROP_FPS) or 20
    wr = cv2.VideoWriter(str(out_path), cv2.VideoWriter_fourcc(*"mp4v"), fps_in, (W, H))
    Track._next = 0
    tracker = SignTracker()
    health = PerceptionHealth(dark_thr=BR_LO * 0.7, blur_thr=BL_LO * 0.1, fail_thr=BR_LO * 0.35)   # thresholds are tunable; zoomed demo frames are soft
    log = [("frame", "time_s", "track_id", "class", "confidence", "x1", "y1", "x2", "y2", "distance_m", "map_check", "health")]
    timeline, preview, f = [], [], 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        _, br, bl = quality(frame)
        state = health.update(br, bl)
        t0 = time.perf_counter()
        dets = detector(frame, thr)
        tracks = tracker.step(dets)
        lat = (time.perf_counter() - t0) * 1000
        warn = False
        for t in tracks:
            cls, conf = t.label
            name = TARGET_CLASSES[cls]
            d_new = camera.distance(t.box[3] - t.box[1], sign_h)
            t.dist = d_new if t.dist is None else 0.7 * t.dist + 0.3 * d_new       # smooth the estimate
            check = plausibility(speed_of(name), map_limit) if speed_of(name) else ""
            if state != "FAILSAFE":
                x1, y1, x2, y2 = map(int, t.box)
                col = PALETTE[cls]
                txt = f"#{t.id} {name} {conf * 100:.0f}% {t.dist:.1f}m" + (f" [{check}]" if check else "")
                cv2.rectangle(frame, (x1, y1), (x2, y2), col, 2)
                for c, th in [((0, 0, 0), 3), (col, 1)]:                          # dark outline, then coloured text
                    cv2.putText(frame, txt, (x1, max(15, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, c, th, cv2.LINE_AA)
                warn = warn or (cls == STOP and t.dist < warn_dist)
            log.append((f, round(f / fps_in, 3), t.id, name, round(conf, 3), *map(int, t.box), round(t.dist, 2), check, state))
        scol = {"NOMINAL": (0, 200, 0), "DEGRADED": (0, 165, 255), "FAILSAFE": (0, 0, 255)}[state]
        cv2.putText(frame, f"{state}: {health.ACTIONS[state]}", (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, scol, 2, cv2.LINE_AA)
        if warn:
            cv2.putText(frame, "STOP AHEAD", (W // 2 - 110, 70), cv2.FONT_HERSHEY_SIMPLEX, 1.1, (0, 0, 255), 3, cv2.LINE_AA)
        wr.write(frame)
        timeline.append({"frame": f, "brightness": br, "blur": bl, "state": state, "raw_dets": len(dets), "tracks": len(tracks), "latency_ms": lat})
        if f % 30 == 0:
            preview.append(frame.copy())
        f += 1
    cap.release(); wr.release()
    with open(csv_path, "w", newline="") as fh:
        csv.writer(fh).writerows(log)
    return pd.DataFrame(timeline), preview


DEMO_IN = build_demo_video(WORK / "demo_input.mp4")
PIPE_DET = VARIANTS.get(DEPLOY_NAME, DET_MAIN)
CAM = Camera.from_hfov(640, 640, 60)                          # synthetic video: field of view is an assumption here
TL, preview = run_pipeline(DEMO_IN, PIPE_DET, OP_THR, CAM, WORK / "pipeline_raw.mp4", WORK / "detections_log.csv")
print(f"{len(TL)} frames | median latency {TL.latency_ms.median():.1f} ms | states:", dict(TL.state.value_counts()))
show_grid(preview[:8], cols=4, size=3.6)

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(11, 6), sharex=True)
ax[0].plot(TL.frame, TL.brightness, color="k"); ax[0].set_ylabel("brightness"); ax[0].axhline(BR_LO, color="gray", ls=":")
ax[1].plot(TL.frame, TL.raw_dets, label="raw detections", alpha=.6); ax[1].plot(TL.frame, TL.tracks, label="confirmed tracks", lw=2); ax[1].legend(fontsize=8); ax[1].set_ylabel("count")
cmap = {"NOMINAL": "tab:green", "DEGRADED": "tab:orange", "FAILSAFE": "tab:red"}
for st, sub in TL.groupby("state"):
    ax[2].scatter(sub.frame, [st] * len(sub), c=cmap[st], s=18)
ax[2].set_xlabel("frame"); ax[2].set_ylabel("health state")
plt.suptitle("Pipeline timeline: image quality, detection stability, health state", y=0.99); finish_plot("pipeline_timeline")

In [ ]:
try:
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(WORK / "pipeline_raw.mp4"), "-vcodec", "libx264", "-pix_fmt", "yuv420p", str(WORK / "pipeline.mp4")], check=True)
    from IPython.display import Video
    display(Video(str(WORK / "pipeline.mp4"), embed=True, width=480))
except Exception as e:
    print("ffmpeg conversion unavailable:", e, "-> download", WORK / "pipeline_raw.mp4")
pd.read_csv(WORK / "detections_log.csv").head(12)

**Check yourself (Step 5)**
1. A sign is 20 px tall in the image and your box is off by 2 px. Roughly how large is the distance error? And for a sign 80 px tall?
2. Why does a track need 3 hits before it is confirmed? What does that cost at 15 frames per second?
3. The camera sees nothing for 10 seconds on a dark road. Does that mean there are no signs? What should the system do?

# Step 6 — Monitoring and MLOps
**What you will learn:** how a model is watched and improved after release, and how every release stays traceable.

Shipping is the *start* of the model's life. The world drifts (new seasons, cities, sign designs, camera ageing), so production systems
1. **monitor** the inputs and outputs without labels (drift detection),
2. run in **shadow mode** and collect disagreements and uncertain cases,
3. feed those back into labelling → retraining (the **data flywheel**, back to Step 2),
4. keep a **registry**: every model is traceable to its data, metrics and gate decision, promotion is controlled, rollback is one call.

## 6.1 Drift monitoring without labels
We replay the "field" as 6 weekly batches. From week 5 on the images are **darker and noisier** (a winter-evening shift the model never saw).
Labels are not available in the real field, so the monitor only looks at **inputs** (brightness, sharpness) and **outputs** (confidence of detections), compared to the training reference with the
**Population Stability Index** (PSI: < 0.1 stable, 0.1–0.25 moderate, > 0.25 major shift; the alert limit is raised automatically when a weekly window is too small to trust).
In this simulation the truth *is* known, so you can check that the label-free alarm really tracks the real performance drop.

**What to check:** in which week does the alarm fire, and in which week does the true F1 drop? Does any signal fire too late or not at all?

In [ ]:
def psi(ref, cur, bins=5):
    edges = np.unique(np.percentile(ref, np.linspace(0, 100, bins + 1)))
    edges[0], edges[-1] = -np.inf, np.inf
    r, c = (np.clip(np.histogram(x, edges)[0] / len(x), 1e-3, None) for x in (ref, cur))
    return float(np.sum((c - r) * np.log(c / r)))


def psi_limit(ref, n, floor=0.25, trials=200):
    """Alert limit = max(0.25, 99th percentile of the PSI you get from pure sampling noise with n samples).
    Small windows are noisy: without this correction a monitor cries wolf every week."""
    r = np.random.RandomState(SEED)
    return max(floor, float(np.percentile([psi(ref, r.choice(ref, n, replace=True)) for _ in range(trials)], 99)))


def dusk(img):
    return degrade(degrade(img, "brightness", 0.45), "noise", 8)


FIELD_DET = PIPE_DET
pool = ITEMS["val"] + ITEMS["test"]
random.Random(SEED).shuffle(pool)
N_WEEKS, SHIFT_FROM = 6, 4
weeks = np.array_split(np.arange(len(pool)), N_WEEKS)

ref_bright = IMGS[IMGS.split == "train"].brightness.values
ref_blur = IMGS[IMGS.split == "train"].blur.values
ref_conf = np.array([p["conf"] for r in PACK["candidate"]["val"] for p in r["preds"] if p["conf"] >= OP_THR])

FIELD_RECS, rows = [], []
for w, idx in enumerate(weeks, start=1):
    recs = []
    for i in idx:
        it = pool[i]
        img = cv2.imread(it["path"])
        if w >= SHIFT_FROM:
            img = dusk(img)
        _, b, l = quality(img)
        recs.append({"path": it["path"], "gts": it["gts"], "preds": FIELD_DET(img, 0.01), "brightness": b, "blur": l, "week": w})
    FIELD_RECS += recs
    br, bl = [r["brightness"] for r in recs], [r["blur"] for r in recs]
    confs = np.array([p["conf"] for r in recs for p in r["preds"] if p["conf"] >= OP_THR])
    m = metrics_from(per_image_counts(recs, OP_THR))
    pb, pl = psi(ref_bright, br), psi(ref_blur, bl)
    pc = psi(ref_conf, confs) if (len(confs) >= 10 and len(ref_conf) >= 10) else float("nan")
    alert = pb > psi_limit(ref_bright, len(br)) or pl > psi_limit(ref_blur, len(bl)) or (pc == pc and pc > psi_limit(ref_conf, len(confs)))
    rows.append({"week": w, "psi_brightness": pb, "psi_blur": pl, "psi_conf": pc, "ALERT": bool(alert),
                 "ks_p_brightness": float(stats.ks_2samp(ref_bright, br).pvalue), "dets_per_img": len(confs) / len(recs),
                 "mean_conf": float(confs.mean()) if len(confs) else float("nan"),
                 "true_recall": m["recall"], "true_f1": nan0(m["f1"])})
DRIFT = pd.DataFrame(rows).set_index("week")
display(DRIFT.round(3))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.4))
for c in ["psi_brightness", "psi_blur", "psi_conf"]:
    ax[0].plot(DRIFT.index, DRIFT[c], marker="o", label=c)
ax[0].axhline(0.25, color="r", ls="--"); ax[0].set_title("Label-free monitors (PSI vs training)")
ax[1].plot(DRIFT.index, DRIFT.true_f1, marker="o", color="k", label="true F1 (needs labels)"); ax[1].plot(DRIFT.index, DRIFT.true_recall, marker="s", color="tab:red", label="true recall")
ax[1].set_ylim(0, 1); ax[1].set_title("Real performance (simulation only)")
for a in ax:
    a.axvspan(SHIFT_FROM - 0.5, N_WEEKS + 0.5, color="orange", alpha=.15); a.set_xlabel("week"); a.legend(fontsize=8)
finish_plot("drift")
first = DRIFT.index[DRIFT.ALERT][0] if DRIFT.ALERT.any() else None
print("First label-free alert: week", first, "| actual shift starts at week", SHIFT_FROM)

## 6.2 Shadow mode and the labelling queue
The model runs silently next to the "ground truth" source (here: our labels; in a car: the driver's behaviour, other sensors, human audits).
Every **disagreement** and every **uncertain** prediction is queued for human labelling, ranked by priority — this is the data flywheel feeding Step 2.
Look at the top of the queue: are these the images you would want labelled first?

In [ ]:
def build_queue(records, thr, uncertain_band=0.20):
    q = []
    for r in records:
        m = match_image(r["preds"], r["gts"], thr)
        reasons, prio = [], 0
        wrong_gt = set(m["wrong"].values())
        for j, hit in enumerate(m["gt_hit"]):
            if hit:
                continue
            if j in wrong_gt:
                reasons.append("wrong class"); prio += 2
            else:
                reasons.append("model missed " + TARGET_CLASSES[r["gts"][j]["cls"]]); prio += 3 if r["gts"][j]["cls"] == STOP else 2
        for i in m["keep"]:
            if not m["pred_tp"][i] and i not in m["wrong"]:
                reasons.append("model saw a sign that is not labelled"); prio += 2
            elif r["preds"][i]["conf"] < thr + uncertain_band:
                reasons.append("low confidence"); prio += 1
        if reasons:
            prio += int(band_brightness(r["brightness"]) == "dark")
            q.append({"path": r["path"], "week": r.get("week"), "priority": prio, "reasons": "; ".join(sorted(set(reasons)))})
    return pd.DataFrame(q).sort_values("priority", ascending=False).reset_index(drop=True)


QUEUE = build_queue(FIELD_RECS, OP_THR)
QUEUE.to_csv(WORK / "labeling_queue.csv", index=False)
print(f"{len(QUEUE)} of {len(FIELD_RECS)} field images queued for labelling (saved to labeling_queue.csv)")
display(QUEUE.groupby("week").size().rename("queued images per week").to_frame().T)
top = QUEUE.head(8)
display(top)
show_grid([cv2.imread(p) for p in top.path], [f"prio {r.priority}: {r.reasons[:34]}" for r in top.itertuples()], cols=4, size=3.2)

## 6.3 Model registry and controlled promotion
A model is not "best.pt in somebody's Drive". It has an **ID**, the **hash of its weights**, the **dataset ID** it was trained on, its **metrics** and the **gate decision**.
Promotion to production is allowed only if the gate said SHIP — anything else needs a **written waiver**, which is logged. Rollback is one call.
(In real teams this lives in MLflow / W&B / an internal registry; the principle is the same.)
The cell tries several promotions on purpose, including a blocked one; read each line of the log and check why it was allowed or refused.

In [ ]:
class ModelRegistry:
    def __init__(self, path):
        self.path = Path(path)
        self.d = json.loads(self.path.read_text()) if self.path.exists() else {"models": [], "production": None, "history": [], "log": []}

    def _save(self):
        self.path.write_text(json.dumps(self.d, indent=2, default=str))

    def _log(self, msg):
        self.d["log"].append(f"{time.strftime('%H:%M:%S')}  {msg}")
        self._save()

    def register(self, name, weights, dataset_id, metrics, gate):
        mid = f"m{len(self.d['models']) + 1}-{file_sha(weights, 8)}"
        self.d["models"].append({"id": mid, "name": name, "weights": str(weights), "dataset_id": dataset_id,
                                 "metrics": {k: (None if v is None else round(float(v), 4)) for k, v in metrics.items()},
                                 "gate": gate, "created": time.strftime("%Y-%m-%d %H:%M:%S")})
        self._log(f"registered {mid} ({name}), gate={gate}")
        return mid

    def promote(self, mid, override_reason=None):
        e = next(m for m in self.d["models"] if m["id"] == mid)
        waived = e["gate"] != "SHIP"
        if waived and not override_reason:
            raise PermissionError(f"{mid}: gate decision is '{e['gate']}', promotion blocked (a written waiver is required)")
        if waived:
            e["waiver"] = override_reason
        if self.d["production"]:
            self.d["history"].append(self.d["production"])
        self.d["production"] = mid
        self._log(f"PROMOTED {mid}" + (f" with waiver: {override_reason}" if waived else ""))

    def rollback(self):
        if not self.d["history"]:
            raise RuntimeError("nothing to roll back to")
        bad, self.d["production"] = self.d["production"], self.d["history"].pop()
        self._log(f"ROLLBACK {bad} -> {self.d['production']}")


(WORK / "registry.json").unlink(missing_ok=True)
REG = ModelRegistry(WORK / "registry.json")
if WEIGHTS_V1:
    v1_id = REG.register("v1 (deployed)", WEIGHTS_V1, DATASET_ID, {k: PACK["v1"]["req"].get(k) for k in ["stop_recall", "other_min_recall", "bg_fa_per_100"]}, "LEGACY")
    REG.promote(v1_id, override_reason="initial deployment of the legacy baseline")
cand_id = REG.register("candidate", WEIGHTS_MAIN, DATASET_ID, PACK["candidate"]["req"], GATE_DECISION)
try:
    REG.promote(cand_id)
    print("candidate promoted to production")
except PermissionError as e:
    print("BLOCKED ->", e)
    print("(To ship anyway an engineer must write a waiver: REG.promote(cand_id, override_reason='...') - and it is logged forever.)")
print("\n".join(REG.d["log"]))
print("production model:", REG.d["production"])

## 6.4 Simulation evidence: distance against ground truth (requirement R6)
Simulation gives what the dataset cannot: **ground-truth distance** and **controlled weather / lighting**. It is the only evidence for R6 in this notebook.
The results come from `webots_demo/` (Webots, open source, ~260 MB): a car drives past all 6 sign types plus 3 look-alike decoys (Speed Limit 20/70, turn left)
in 6 conditions (noon, overcast, dusk, night with headlights, fog, dense fog). The simulator knows where each sign is, so every frame gives the true distance.
Upload the generated `sim_results/` folder here (or put it on Drive). The CARLA scripts in `carla_demo/` write the same files and also work.
Without such a folder this section is skipped and R6 stays NOT MEASURED, which is the honest answer.

**What to check:** how the error grows with distance (far signs are only a few pixels tall), and which condition hurts detection most.
Keep in mind the sim-to-real gap: rendered signs are cleaner than real photos, so treat simulation as *necessary* evidence, not *sufficient*.

In [ ]:
SIM_CANDIDATES = [Path("/content/sim_results"), Path("/content/drive/MyDrive/traffic_sign_data/sim_results"), Path("webots_demo/sim_results"), Path("sim_results"),
                  Path("/content/carla_results"), WORK / "carla_results", Path("carla_results"), Path("/content/drive/MyDrive/carla_results")]
CARLA_DIR = next((p for p in SIM_CANDIDATES if (p / "distance_log.csv").exists()), None)
if CARLA_DIR is None:
    print("No simulation results found. Upload the sim_results (or carla_results) folder to /content and re-run this cell.")
else:
    dl, ws = (pd.read_csv(CARLA_DIR / f) if (CARLA_DIR / f).exists() else None for f in ("distance_log.csv", "weather_sweep.csv"))
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
    if dl is not None and len(dl):
        dl["abs_err_pct"] = (dl.est_dist_m - dl.gt_dist_m).abs() / dl.gt_dist_m * 100
        ax[0].scatter(dl.gt_dist_m, dl.est_dist_m, s=10, alpha=.5); lim = [0, max(dl.gt_dist_m.max(), dl.est_dist_m.max()) * 1.05]
        ax[0].plot(lim, lim, "k--", lw=1); ax[0].set_xlabel("ground-truth distance (m)"); ax[0].set_ylabel("estimated (m)"); ax[0].set_title(f"Pinhole distance vs simulator ground truth ({CARLA_DIR.name})")
        med = PACK["candidate"]["req"]["distance_err_pct"] = float(dl.abs_err_pct.median())
        print(f"median distance error {med:.1f}%  |  p90 {dl.abs_err_pct.quantile(.9):.1f}%  (R6 limit {REQS[5].target}%)")
    if ws is not None and len(ws):
        ax[1].bar(ws.weather, ws.detection_rate, color="tab:blue"); ax[1].set_ylim(0, 1); ax[1].set_ylabel("detection rate of visible signs")
        ax[1].tick_params(axis="x", rotation=60); ax[1].set_title("Detection rate of visible signs by condition (simulation)")
        display(ws.round(3))
    finish_plot("simulation")

# Final scorecard
Everything measured in this notebook, against the requirements written in Step 1. **Anything that is not PASS is a work item, not a failure** — this is what an honest status report looks like.
Read it as if you had to sign it: for each line, which section produced the evidence, and how much would you trust it?

In [ ]:
SCORE = pd.DataFrame([{"req": r.rid, "requirement": r.text, "measured": v, "limit": f"{r.op} {r.target}{r.unit}",
                       "status": "NOT MEASURED" if ok is None else ("PASS" if ok else "FAIL")}
                      for r, v, ok in req_rows(PACK["candidate"]["req"])]).set_index("req")
display(SCORE)
GATE_DECISION, _ = release_gate(PACK["candidate"], PACK.get("v1"), verbose=False)
print(f"Release gate decision: {GATE_DECISION}")
print(f"Dataset {DATASET_ID} | near-duplicates between test and train: {LEAK_TEST_PCT:.0f}% | deployed variant: {PACK['candidate'].get('deploy_variant')}")
SCORE.to_csv(WORK / "scorecard.csv")

## The loop
```
  Requirements ──► Data ──► Training & validation ──► Gate ──► Deployment ──► Perception stack ──► Fleet / road
       ▲                                                                                              │
       └──────────── new requirements ◄── labelling queue ◄── drift alerts / shadow-mode disagreements ◄┘
```
**Key ideas**
1. A requirement is testable; "good accuracy" is not.
2. The data decides the ceiling; audit it, slice it, version it, keep the test set honest.
3. Report by slice with uncertainty; gate on the worst case, not the average.
4. Speed is a requirement — and every optimisation must be re-validated.
5. A detector is a component; tracking, calibration, cross-checks and fallbacks make it a system.
6. After release, monitor, collect, retrain: the loop never closes.

**Check yourself (Step 6)**
1. Why can't you monitor accuracy directly in the field, and what do you monitor instead?
2. Someone wants to promote a model whose gate said NO-SHIP "just for a test drive". What does the registry require, and why?
3. The final scorecard: which line is the weakest evidence, even if it says PASS?

**Going further with your own car:** write your own requirements and ODD, collect and label images from *your* camera, run this slice table and gate automatically on every new model,
calibrate the camera, and extend the Webots world in `webots_demo/` (curves, other cars, rain) for closed-loop tests. Good exercises in this notebook: retrain with `USE_DATASETS = ["a"]` and compare,
change the cost table and watch the threshold move, or add a new degradation to the robustness sweep.